In [ ]:
# ================================================================
# Powell, CS7367 (Machine Vision): Fruit Detection & Simulated Robotic
# Grasping, v14 + depth-camera localization. Single-cell version: paste
# everything into one Colab cell and run. Colab runtime: GPU (L4), High-RAM.
#
# Order of work: setup -> definitions -> camera sanity check ->
# dataset generation -> training -> end-to-end trials (all 3 classes)
# -> detection-only diagnostic -> grasp-only comparison -> summary
# -> depth-camera localization check -> depth-localization trials.
# Expect about 4-5 hours in total, mostly training. Everything is saved
# to Google Drive. After a disconnect, re-run the cell: finished training
# is reused, and finished depth-localization runs are reused. To skip
# v14 stages already done, set their RUN_* switches to False.
# ================================================================
!pip install -q pybullet ultralytics

# ── Run switches ──────────────────────────────────────────────────
# If Colab disconnects after training finished, just re-run this cell:
# with REUSE_TRAINED_MODEL = True, generation and training are skipped
# when models/fruit_detector_v14.pt already exists on Google Drive.
# If it disconnects mid-training, re-running resumes from the last epoch.
RUN_SANITY_CHECK    = True
REUSE_TRAINED_MODEL = True
RESUME_TRAINING     = True
RUN_TRIALS          = True
RUN_DIAGNOSTIC      = True
RUN_GRASP_ONLY      = True
RUN_SUMMARY         = True
# Depth camera (see the DEPTH-CAMERA LOCALIZATION section at the end)
RUN_LOC_VALIDATION  = True    # estimate vs. truth on 10 scenes per class
RUN_LOC_TRIALS      = True    # end-to-end trials aimed from depth
DEPTH_NOISE_STD     = 0.0     # m; 0.0 = ideal depth. e.g. 0.003 = 3 mm noise
DEPTH_DROPOUT       = 0.0     # fraction of depth pixels lost, e.g. 0.02

# ── Working folders ────────────────────────────────────────────────
# Models, training runs, results and sample images go to Google Drive,
# so a Colab disconnect does not lose a multi-hour training run.
# The generated dataset (thousands of small PNGs) stays on the fast local
# disk; it is regenerated deterministically (seed 42) if ever needed.
import os
USE_DRIVE = True    # keeps the trained model if Colab disconnects
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/CS7367_fruit_grasp_v14"
else:
    WORKDIR = "/content/CS7367_fruit_grasp_v14"
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
os.environ["FRUIT_DATA_DIR"] = "/content/data/detect_v14"   # local disk
print("Working directory:", os.getcwd())
print("Dataset directory:", os.environ["FRUIT_DATA_DIR"])

# ================================================================
# v14 — CHANGES FROM v13
# Evidence below was measured by running this pipeline's scene, camera
# and grasp code in a local PyBullet replica (no detector involved).
# ================================================================
# V14-1 CAMERA AIM. v13's eval camera looked at z=0.03 (the floor under
#   the table); the table top is at z=0.626. Only table x~0.86-1.00 m was
#   in frame, while fruit spawn at x 0.48-0.68 m. Replica: the target was
#   visible in 5 of 25 trials that passed the zone check. v14 aims every
#   pose at the table top: 40 of 40 targets visible.
# V14-2 ROBOT PLACEMENT. v13 loaded the Panda base at z=0, inside the
#   table; the arm passed ~10 cm through the table top. v14 mounts the
#   base on the table and starts from a standard ready pose. Same v13
#   grasp procedure, banana grasp-only test (seed 3003, 50 scenes; report
#   Table VI): 6 of 39 attempts (11 targets knocked off first) -> 42/50.
# V14-3 TRIAL GATE. The position-based zone check is replaced by a direct
#   visibility check on the target's segmentation mask (>= 30 px).
# V14-4 GRASP. grasp_mode="v14": fingertip-centred IK target, yaw aligned
#   to the fruit, waypointed motion, coupled fingers, and a grasp only
#   counts if BOTH fingers touch the target (v13's constraint attached
#   whether or not the fingers touched anything). Grasp-only test
#   (seed 3003, 50 scenes per class; report Table VI): banana 44/50,
#   orange 46/50, pear 30/50. grasp_mode="v13" is kept for the
#   before/after comparison.
# V14-5 LOGGING. Every evaluation is seeded; every trial ends in exactly
#   one outcome (v13 counted bad_localization per detection, not per
#   trial); per-trial results are saved as JSON in results_v14/.
# V14-6 HOUSEKEEPING. Video camera re-aimed; stale "orange" caption and
#   A100 message fixed; output paths bumped to v14.
# UNCHANGED: dataset generator logic and class sampling, augmentation,
#   training hyper-parameters, detector, conf/IoU thresholds. Before/after
#   differences therefore come from the fixes above.
# ================================================================
import os
import cv2
import json
import shutil
import random
import argparse
import numpy as np
import torch
from dataclasses import dataclass
from pathlib import Path
from glob import glob
from collections import Counter
from tqdm import trange
import pybullet as p
import pybullet_data
from ultralytics import YOLO
# ============================================================
# GPU validation
# ============================================================
def validate_gpu():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "No CUDA GPU detected. "
            "In Colab: Runtime -> Change runtime type -> select a GPU (L4 used for this project) -> Save, "
            "then Runtime -> Restart runtime and re-run."
        )
    gpu_name  = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[GPU] Using: {gpu_name} | VRAM: {total_mem:.1f} GB")
    print(f"[GPU] PyTorch version: {torch.__version__}")
    print(f"[GPU] CUDA version:    {torch.version.cuda}")
    return 0
DEVICE = validate_gpu()
# ============================================================
# Config
# ============================================================
@dataclass
class Config:
    fruit_classes: tuple = ("orange", "pear", "banana")
    # ── FIX 6: Switch grasp target to banana ──────────────────────
    # v12 evidence — train boxes were pear:75, banana:516, orange:80.
    # Banana had 6.5x the training examples of orange and was the
    # only class that ever cleared conf_thresh during trials (0.81,
    # 0.94), even in scenes where orange was the requested target.
    # Targeting banana uses the class the detector already knows
    # well instead of retraining to fix orange's imbalance.
    target_class:  str   = "banana"
    cam_w: int = 640
    cam_h: int = 480
    obj_id_mask: int = (1 << 24) - 1
    outdir:       str = os.environ.get("FRUIT_DATA_DIR", "data/detect_v14")
    model_path:   str = "models/fruit_detector_v14.pt"
    classes_path: str = "models/class_names_v14.json"
    samples_dir:  str = "samples_v14"
    train_episodes: int = 800
    val_episodes:   int = 150
    fruits_per_scene_min: int = 4
    fruits_per_scene_max: int = 8
    distractors_per_scene_max: int = 3
    views_per_scene_train: int = 4
    views_per_scene_val:   int = 3
    min_mask_pixels: int = 5
    min_area:        int = 10
    min_box_side:    int = 3
    bbox_pad:        int = 4
    empty_scene_prob_train: float = 0.00
    empty_scene_prob_val:   float = 0.00
    target_focused_scene_prob_train: float = 0.85
    target_focused_scene_prob_val:   float = 0.80
    class_min_box_quota: int = 30
    seed: int = 42
    # Training hyper-parameters
    epochs:     int = 150
    imgsz:      int = 800
    batch_size: int = 12
    pretrained: str = "yolov8m.pt"
    # Trial detection parameters
    requested:             str   = "banana"   # FIX 6: was "orange"
    trials:                int   = 50
    lift_height_threshold: float = 0.20
    conf_thresh:           float = 0.05
    save_samples:          int   = 12
    trial_match_iou_thresh: float = 0.01
    # Trial spawn zone for the requested (target) fruit, any class.
    # Values unchanged from v13 (renamed from trial_banana_* because
    # v14 runs trials for all three classes).
    trial_target_x_min: float = 0.48
    trial_target_x_max: float = 0.68
    trial_target_y_min: float = -0.12
    trial_target_y_max: float = 0.12
    # ── FIX 1: Verification abort removed entirely ────────────────
    # v11 evidence — Trials 03, 04, 36 had final heights of
    # 0.199m, 0.345m, 0.199m but were aborted by verify at -87mm,
    # -89mm, -0.4mm. The dip is a physics transient during arm
    # acceleration at the start of the nudge, not a grasp failure.
    # Trial 04 at 0.345m would have been a clear success.
    # Verification is killing more winning trials than it catches.
    # v12 removes the abort path entirely — attempt the full lift
    # unconditionally, measure only at the end.
    # Kept: post_constraint_settle (60 steps) lets physics settle
    # after constraint attachment before starting the lift.
    post_constraint_settle: int = 60   # settle after constraint, before lift
    # ── FIX 2: Constraint uses both-zero frame positions ─────────
    # v11 Trials 25, 28, 32 passed verification but collapsed during
    # full lift (-0.319m, -0.292m, -0.158m). The parentFramePosition
    # offset rotates with the EE during the lift arc, creating
    # geometry instability. Setting both parent and child frame
    # positions to [0,0,0] anchors each body at its own origin —
    # stable regardless of arm configuration during lift.
    constraint_parent_offset: float = 0.0  # both frames at body origin
    constraint_child_offset:  float = 0.0  # child frame also at origin
    # ── FIX 3: Reduce trial scene density ────────────────────────
    # v11 evidence — 38/50 no-detection despite center-zone spawn.
    # 200-step settle (FIX 5 v11) made no difference, eliminating
    # the "banana rolling after settle" hypothesis. Trials 09, 16
    # show pear detections at 0.84–0.97 confidence — other fruits
    # are landing in center zone and occluding/displacing the banana.
    # Reducing from 6 to 3 fruits per trial scene and increasing
    # minimum separation from 0.10m to 0.15m dramatically reduces
    # inter-fruit collision probability.
    fruits_in_scene:  int   = 3     # was 6
    trial_min_dist:   float = 0.15  # was 0.10
    # ── FIX 4: Post-settle banana position check ──────────────────
    # After physics settling, verify the banana is still within the
    # camera-visible center zone before running detection. If it has
    # rolled outside due to collisions, skip directly to next trial
    # rather than burning a trial on guaranteed no-detection.
    # Zone is slightly wider than spawn zone to allow small roll.
    # ── FIX 7: Widen the check zone ────────────────────────────────
    # v12 evidence — 20/50 trials (40%) were thrown away here before
    # detection ever ran, on top of the 30 no-detection fails. The
    # old zone was only +/-0.04m past the spawn zone on each side.
    # Widening it gives the target more room to settle/roll without
    # burning a trial, while staying inside the camera's visible
    # field at the "eval" pose used for trials.
    trial_check_x_min: float = 0.38   # was 0.44
    trial_check_x_max: float = 0.78   # was 0.72
    trial_check_y_min: float = -0.22  # was -0.16
    trial_check_y_max: float = 0.22   # was 0.16
    # ── FIX 5: More lift steps + post-lift settle ─────────────────
    # Trials 03 and 36 failed at exactly 0.199m — 1mm short of 0.20m
    # threshold. The IK target is table_top_z + threshold + 0.30m
    # but 400 steps was insufficient to fully converge.
    # Raising to 500 steps gives IK more convergence time.
    # Adding 50-step pure settle before final measurement lets
    # any residual physics oscillation die out before measuring.
    lift_above_threshold: float = 0.30   # lift target = threshold + 0.30
    lift_steps:           int   = 500    # was 400
    post_lift_settle:     int   = 50     # NEW: settle before final measure
    # Grasping parameters kept from v11
    ik_settle_steps:   int = 100
    grip_close_steps:  int = 200   # close BEFORE constraint
    post_grip_settle:  int = 50    # settle after close, before constraint
    # Trial settle
    trial_settle_steps: int = 200
    # Video output
    video_fps:   int = 30
    video_path:  str = "successful_grasp.mp4"
    video_cam_w: int = 640
    video_cam_h: int = 480
    # ================================================================
    # v14 additions (see notebook header for rationale and evidence)
    # ================================================================
    # V14-1 camera: all camera poses are now aimed at the table top.
    # v13 aimed at z=0.03 (the floor, under the table); the frame only
    # covered table x ~0.86-1.00 m while fruit spawn at x 0.48-0.68 m.
    table_top_z_nominal: float = 0.626   # PyBullet table.urdf top (measured)
    # V14-2 robot placement: v13 loaded the Panda base at z=0, i.e. on
    # the floor *inside* the table; the arm passed ~10 cm through the
    # table top. v14 mounts the base on the table top.
    robot_on_table: bool = True
    # V14-3 trial gating: v13's position-based zone check is replaced
    # by a direct visibility check on the target's segmentation mask.
    use_zone_check:     bool = False
    min_visible_pixels: int  = 30
    # V14-4 grasp: "v14" = aim the panda_grasptarget link (between the
    # fingertips) at the object, yaw-aligned to the object, waypointed
    # motion, and a constraint that preserves the in-hand pose.
    # "v13" = original procedure, kept for the before/after comparison.
    grasp_mode:             str   = "v14"
    require_finger_contact: bool  = True    # both fingers must touch target
    pregrasp_clearance:     float = 0.20    # m above object before descent
    grasp_min_clearance:    float = 0.012   # m, grasp point never below this
    lift_target_above:      float = 0.15    # lift goal = threshold + this
    move_waypoints:         int   = 10
    grasp_yaw_offset:       float = float(np.pi / 2)  # fingers close across the long axis
    banana_grasp_link:      int   = 1     # link index of banana sphere to grasp
    pear_grasp_drop:        float = 0.0   # m below pear top-sphere centre (0.015 and 0.03 tested: no better)
    couple_fingers:         bool  = True  # gear-couple the two fingers
    finger_close_velocity:  float = 0.05  # m/s max finger speed while closing
    constraint_max_force:   float = 200.0
    # V14-5 reproducibility: every evaluation is seeded.
    trial_seed:      int = 1001
    diag_seed:       int = 2002
    grasp_test_seed: int = 3003
    results_dir:     str = "results_v14"
CFG = Config()
CLASS_TO_ID = {c: i for i, c in enumerate(CFG.fruit_classes)}
# ============================================================
# Utilities
# ============================================================
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
def reset_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    ensure_dir(path)
def ensure_sample_dirs(samples_dir: str):
    ensure_dir(samples_dir)
    ensure_dir(os.path.join(samples_dir, "generate"))
    ensure_dir(os.path.join(samples_dir, "run"))
def write_dataset_yaml(outdir: str, class_names):
    yaml_path  = os.path.join(outdir, "data.yaml")
    abs_outdir = os.path.abspath(outdir)
    with open(yaml_path, "w") as f:
        f.write(f"path: {abs_outdir}\n")
        f.write("train: images/train\n")
        f.write("val:   images/val\n")
        f.write("names:\n")
        for i, cls in enumerate(class_names):
            f.write(f"  {i}: {cls}\n")
    return yaml_path
def rand_color(base_rgb, jitter=0.08):
    rgb = np.array(base_rgb, dtype=np.float32)
    rgb += np.random.uniform(-jitter, jitter, size=3)
    rgb = np.clip(rgb, 0.0, 1.0)
    return [float(rgb[0]), float(rgb[1]), float(rgb[2]), 1.0]
def clip_bbox(bb, w, h):
    x1, y1, x2, y2 = bb
    x1 = int(np.clip(x1, 0, w - 1))
    y1 = int(np.clip(y1, 0, h - 1))
    x2 = int(np.clip(x2, 0, w - 1))
    y2 = int(np.clip(y2, 0, h - 1))
    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2
def pad_bbox(bb, pad, w, h):
    x1, y1, x2, y2 = bb
    return clip_bbox((x1 - pad, y1 - pad, x2 + pad, y2 + pad), w, h)
def xyxy_to_yolo_line(bb, cls_id, img_w, img_h):
    x1, y1, x2, y2 = bb
    bw = x2 - x1 + 1
    bh = y2 - y1 + 1
    cx = x1 + bw / 2.0
    cy = y1 + bh / 2.0
    return (f"{cls_id} {cx / img_w:.6f} {cy / img_h:.6f} "
            f"{bw / img_w:.6f} {bh / img_h:.6f}")
def draw_bboxes(rgb, boxes_with_labels, title=None):
    """Accepts RGB image, returns BGR image ready for cv2.VideoWriter."""
    img = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    colors = {
        "orange":     (0, 140, 255),
        "pear":       (0, 220, 0),
        "banana":     (0, 255, 255),
        "distractor": (255, 0, 255),
    }
    for bb, label in boxes_with_labels:
        x1, y1, x2, y2 = map(int, bb)
        c = colors.get(label, (255, 255, 255))
        cv2.rectangle(img, (x1, y1), (x2, y2), c, 2)
        cv2.putText(img, label, (x1, max(14, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, c, 2, cv2.LINE_AA)
    if title:
        cv2.putText(img, title, (10, 24),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    return img  # BGR
# ============================================================
# Video rendering helpers
# ============================================================
def render_video_frame(w, h, cam_pos, tgt, fov=52.0,
                       detections=None, label=None):
    view = p.computeViewMatrix(cam_pos, tgt, [0, 0, 1])
    proj = p.computeProjectionMatrixFOV(fov=fov, aspect=w / h,
                                        nearVal=0.02, farVal=2.0)
    try:
        img = p.getCameraImage(w, h, view, proj,
                               renderer=p.ER_BULLET_HARDWARE_OPENGL)
    except Exception:
        img = p.getCameraImage(w, h, view, proj,
                               renderer=p.ER_TINY_RENDERER)
    rgba  = np.reshape(img[2], (h, w, 4))
    frame = cv2.cvtColor(rgba[:, :, :3].astype(np.uint8), cv2.COLOR_RGB2BGR)
    if detections:
        colors = {"banana": (0, 255, 255), "orange": (0, 140, 255),
                  "pear": (0, 220, 0)}
        for conf, cls_name, bb in detections:
            x1, y1, x2, y2 = map(int, bb)
            c = colors.get(cls_name, (255, 255, 255))
            cv2.rectangle(frame, (x1, y1), (x2, y2), c, 2)
            cv2.putText(frame, f"{cls_name} {conf:.2f}",
                        (x1, max(14, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, c, 2, cv2.LINE_AA)
    if label:
        cv2.putText(frame, label, (10, h - 14),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2,
                    cv2.LINE_AA)
    return frame  # BGR
def open_video_writer(path, w, h, fps):
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(path, fourcc, fps, (w, h))
    if not writer.isOpened():
        raise RuntimeError(f"Could not open VideoWriter for path: {path}")
    return writer
# ============================================================
# PyBullet setup
# ============================================================
def connect_direct():
    if p.isConnected():
        try:
            p.disconnect()
        except Exception:
            pass
    p.connect(p.DIRECT)
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -9.81)
    p.setTimeStep(1.0 / 240.0)
    p.setRealTimeSimulation(0)
def reset_world():
    p.resetSimulation()
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -9.81)
    p.setTimeStep(1.0 / 240.0)
    p.setRealTimeSimulation(0)
def randomize_scene_appearance(table_id, plane_id):
    table_color = rand_color([0.48, 0.32, 0.18], jitter=0.10)
    plane_gray  = float(np.random.uniform(0.72, 0.94))
    plane_color = [plane_gray, plane_gray, plane_gray, 1.0]
    p.changeVisualShape(table_id, -1, rgbaColor=table_color)
    p.changeVisualShape(plane_id, -1, rgbaColor=plane_color)
PANDA_READY_POSE = [0.0, -0.785, 0.0, -2.356, 0.0, 1.571, 0.785]
def load_scene():
    plane_id = p.loadURDF("plane.urdf")
    table_id = p.loadURDF("table/table.urdf", [0.5, 0.0, 0.0],
                           useFixedBase=True)
    table_top_z = p.getAABB(table_id)[1][2]
    # V14-2: mount the robot on the table top (v13: base at z=0, inside the table)
    base_z   = table_top_z if CFG.robot_on_table else 0.0
    robot_id = p.loadURDF("franka_panda/panda.urdf", [0, 0, base_z],
                           useFixedBase=True)
    if CFG.robot_on_table:
        arm, fingers, _ = find_panda_indices(robot_id)
        for j, q in zip(arm, PANDA_READY_POSE):
            p.resetJointState(robot_id, j, q)
        for j in fingers:
            p.resetJointState(robot_id, j, 0.04)
    randomize_scene_appearance(table_id, plane_id)
    return robot_id, table_id, table_top_z
# ============================================================
# Camera and rendering
# ============================================================
def get_camera_pose(mode="eval"):
    # V14-1: every pose now looks at the table top (T), not the floor.
    # v13 eval pose was cam (1.0, -0.05, 0.72) -> target (0.58, 0.00, 0.03).
    T = CFG.table_top_z_nominal
    if mode == "eval":
        cam_pos = np.array([1.0, -0.05, T + 0.60], dtype=np.float32)
        tgt     = np.array([0.55, 0.00, T + 0.00], dtype=np.float32)
        fov     = 52.0
    elif mode == "eval_jitter":
        cam_pos = np.array([
            np.random.uniform(0.97, 1.03),
            np.random.uniform(-0.07, 0.02),
            T + np.random.uniform(0.57, 0.63),
        ], dtype=np.float32)
        tgt = np.array([
            np.random.uniform(0.54, 0.57),
            np.random.uniform(-0.02, 0.02),
            T + np.random.uniform(-0.01, 0.02),
        ], dtype=np.float32)
        fov = float(np.random.uniform(50, 54))
    elif mode == "close":   # not used by choose_view_modes; kept for experiments
        cam_pos = np.array([
            np.random.uniform(0.92, 0.98),
            np.random.uniform(-0.08, 0.08),
            T + np.random.uniform(0.42, 0.48),
        ], dtype=np.float32)
        tgt = np.array([
            np.random.uniform(0.56, 0.62),
            np.random.uniform(-0.03, 0.03),
            T + np.random.uniform(0.00, 0.02),
        ], dtype=np.float32)
        fov = float(np.random.uniform(46, 50))
    else:                   # "random"; not used by choose_view_modes
        cam_pos = np.array([
            np.random.uniform(0.95, 1.08),
            np.random.uniform(-0.15, 0.15),
            T + np.random.uniform(0.44, 0.56),
        ], dtype=np.float32)
        tgt = np.array([
            np.random.uniform(0.54, 0.64),
            np.random.uniform(-0.06, 0.06),
            T + np.random.uniform(0.00, 0.03),
        ], dtype=np.float32)
        fov = float(np.random.uniform(50, 58))
    return cam_pos, tgt, fov
def render_camera(w=640, h=480, mode="eval"):
    cam_pos, tgt, fov = get_camera_pose(mode)
    view = p.computeViewMatrix(cam_pos.tolist(), tgt.tolist(), [0, 0, 1])
    proj = p.computeProjectionMatrixFOV(fov=fov, aspect=w / h,
                                        nearVal=0.02, farVal=2.0)
    try:
        img = p.getCameraImage(w, h, view, proj,
                               renderer=p.ER_BULLET_HARDWARE_OPENGL)
    except Exception:
        img = p.getCameraImage(w, h, view, proj,
                               renderer=p.ER_TINY_RENDERER)
    rgba = np.reshape(img[2], (h, w, 4))
    seg  = np.reshape(img[4], (h, w))
    rgb  = rgba[:, :, :3].astype(np.uint8)
    return rgb, seg
# ============================================================
# Segmentation mask helpers
# ============================================================
def seg_mask_for_body_all_links(seg_img, body_id):
    seg_i64 = seg_img.astype(np.int64)
    obj_ids = seg_i64 & CFG.obj_id_mask
    valid   = seg_i64 >= 0
    return valid & (obj_ids == int(body_id))
def bbox_from_mask_union(seg_img, body_id):
    mask        = seg_mask_for_body_all_links(seg_img, body_id)
    pixel_count = int(mask.sum())
    if pixel_count == 0:
        return None, 0
    ys, xs = np.where(mask)
    return (int(xs.min()), int(ys.min()),
            int(xs.max()), int(ys.max())), pixel_count
# ============================================================
# Spawn zone helpers
# ============================================================
def random_table_pos(table_top_z, center_bias=0.0):
    use_center = random.random() < center_bias
    if use_center:
        x = np.random.uniform(0.54, 0.65)
        y = np.random.uniform(-0.09, 0.09)
    else:
        x = np.random.uniform(0.51, 0.67)
        y = np.random.uniform(-0.16, 0.16)
    z = table_top_z + np.random.uniform(0.08, 0.12)
    return [float(x), float(y), float(z)]
def sample_positions_for_objects(table_top_z, fruit_plan, n_distractors,
                                 min_dist=0.085, max_tries=400):
    positions = []
    labels    = list(fruit_plan) + (["distractor"] * n_distractors)
    for lbl in labels:
        center_bias = 0.0
        if lbl == CFG.target_class:
            center_bias = 0.85
        elif lbl in CFG.fruit_classes:
            center_bias = 0.40
        chosen = None
        for _ in range(max_tries):
            pos = random_table_pos(table_top_z, center_bias=center_bias)
            ok  = all(
                np.hypot(pos[0] - prev[0], pos[1] - prev[1]) >= min_dist
                for prev in positions
            )
            if ok:
                chosen = pos
                break
        if chosen is None:
            chosen = random_table_pos(table_top_z, center_bias=0.15)
        positions.append(chosen)
    return positions
# ============================================================
# Trial-time spawn — FIX 3: reduced density + wider separation
# ============================================================
def spawn_trial_scene(table_top_z, fruit_plan,
                      min_dist=None):
    """
    Spawn fruits for a trial scene.
    Target fruit (first) goes into tight center zone.
    Other fruits use wider zone with larger separation (FIX 3).
    Returns list of (fruit_id, class) pairs.
    """
    if min_dist is None:
        min_dist = CFG.trial_min_dist   # FIX 3: 0.15m (was 0.10m)
    placed_positions = []
    spawned          = []
    for idx, cls in enumerate(fruit_plan):
        if idx == 0:
            # Target — guaranteed tight center zone
            chosen = None
            for _ in range(400):
                x = np.random.uniform(CFG.trial_target_x_min,
                                      CFG.trial_target_x_max)
                y = np.random.uniform(CFG.trial_target_y_min,
                                      CFG.trial_target_y_max)
                z = table_top_z + np.random.uniform(0.08, 0.12)
                pos = [float(x), float(y), float(z)]
                ok  = all(
                    np.hypot(pos[0] - p2[0], pos[1] - p2[1]) >= min_dist
                    for p2 in placed_positions
                )
                if ok:
                    chosen = pos
                    break
            if chosen is None:
                chosen = [float(np.random.uniform(0.57, 0.62)),
                          float(np.random.uniform(-0.05, 0.05)),
                          table_top_z + 0.10]
        else:
            # Non-target — wider zone, far from center
            chosen = None
            for _ in range(400):
                pos = random_table_pos(table_top_z, center_bias=0.20)
                ok  = all(
                    np.hypot(pos[0] - p2[0], pos[1] - p2[1]) >= min_dist
                    for p2 in placed_positions
                )
                if ok:
                    chosen = pos
                    break
            if chosen is None:
                chosen = random_table_pos(table_top_z, center_bias=0.0)
        fid = spawn_fruit(cls, chosen)
        placed_positions.append(chosen)
        spawned.append((fid, cls))
    return spawned
# ============================================================
# FIX 4: Post-settle position check
# ============================================================
def target_in_check_zone(body_id):
    """
    Returns True if the target fruit base link is still within the
    camera-visible detection zone after physics settling.
    Prevents burning a trial on a fruit that moved out of view.
    """
    pos, _ = p.getBasePositionAndOrientation(body_id)
    bx, by = pos[0], pos[1]
    return (CFG.trial_check_x_min <= bx <= CFG.trial_check_x_max and
            CFG.trial_check_y_min <= by <= CFG.trial_check_y_max)
# ============================================================
# Fruit spawning
# ============================================================
def spawn_orange(pos):
    r   = float(np.random.uniform(0.032, 0.040))
    col = p.createCollisionShape(p.GEOM_SPHERE, radius=r)
    vis = p.createVisualShape(p.GEOM_SPHERE, radius=r,
                              rgbaColor=rand_color([1.0, 0.55, 0.08], jitter=0.06))
    return p.createMultiBody(
        baseMass=0.05,
        baseCollisionShapeIndex=col,
        baseVisualShapeIndex=vis,
        basePosition=pos,
        baseOrientation=p.getQuaternionFromEuler(
            [0, 0, np.random.uniform(-np.pi, np.pi)]),
    )
def spawn_pear(pos):
    base_r     = float(np.random.uniform(0.037, 0.045))
    top_r      = float(np.random.uniform(0.022, 0.028))
    green1     = rand_color([0.28, 0.78, 0.25], jitter=0.08)
    green2     = rand_color([0.34, 0.88, 0.30], jitter=0.08)
    stem_color = rand_color([0.45, 0.28, 0.10], jitter=0.04)
    col0 = p.createCollisionShape(p.GEOM_SPHERE, radius=base_r)
    vis0 = p.createVisualShape(p.GEOM_SPHERE, radius=base_r, rgbaColor=green1)
    col1 = p.createCollisionShape(p.GEOM_SPHERE, radius=top_r)
    vis1 = p.createVisualShape(p.GEOM_SPHERE, radius=top_r, rgbaColor=green2)
    col2 = p.createCollisionShape(p.GEOM_CYLINDER, radius=0.008, height=0.022)
    vis2 = p.createVisualShape(p.GEOM_CYLINDER, radius=0.008, length=0.022,
                               rgbaColor=stem_color)
    return p.createMultiBody(
        baseMass=0.05,
        baseCollisionShapeIndex=col0,
        baseVisualShapeIndex=vis0,
        basePosition=pos,
        baseOrientation=p.getQuaternionFromEuler(
            [0, 0, np.random.uniform(-np.pi, np.pi)]),
        linkMasses=[0.01, 0.005],
        linkCollisionShapeIndices=[col1, col2],
        linkVisualShapeIndices=[vis1, vis2],
        linkPositions=[[0, 0, 1.32 * base_r],
                       [0, 0, 1.32 * base_r + 0.035]],
        linkOrientations=[[0, 0, 0, 1],
                          p.getQuaternionFromEuler([np.pi / 2, 0, 0])],
        linkInertialFramePositions=[[0, 0, 0], [0, 0, 0]],
        linkInertialFrameOrientations=[[0, 0, 0, 1], [0, 0, 0, 1]],
        linkParentIndices=[0, 1],
        linkJointTypes=[p.JOINT_FIXED, p.JOINT_FIXED],
        linkJointAxis=[[0, 0, 1], [0, 0, 1]],
    )
def spawn_banana(pos):
    scale      = float(np.random.uniform(1.4, 1.8))
    r          = 0.019 * scale
    yellow     = rand_color([1.0, 0.93, 0.10], jitter=0.06)
    stem_brown = rand_color([0.35, 0.22, 0.08], jitter=0.03)
    cols = [p.createCollisionShape(p.GEOM_SPHERE, radius=r) for _ in range(5)]
    viss = [p.createVisualShape(p.GEOM_SPHERE, radius=r, rgbaColor=yellow)
            for _ in range(5)]
    stem_col = p.createCollisionShape(p.GEOM_CYLINDER,
                                      radius=0.009 * scale,
                                      height=0.022 * scale)
    stem_vis = p.createVisualShape(p.GEOM_CYLINDER,
                                   radius=0.009 * scale,
                                   length=0.022 * scale,
                                   rgbaColor=stem_brown)
    yaw = np.random.uniform(-np.pi, np.pi)
    return p.createMultiBody(
        baseMass=0.05,
        baseCollisionShapeIndex=cols[0],
        baseVisualShapeIndex=viss[0],
        basePosition=pos,
        baseOrientation=p.getQuaternionFromEuler([0, 0, yaw]),
        linkMasses=[0.01, 0.01, 0.01, 0.01, 0.004],
        linkCollisionShapeIndices=[cols[1], cols[2], cols[3], cols[4], stem_col],
        linkVisualShapeIndices=[viss[1], viss[2], viss[3], viss[4], stem_vis],
        linkPositions=[
            [0.028 * scale, 0.006 * scale, 0.000],
            [0.056 * scale, 0.018 * scale, 0.000],
            [0.082 * scale, 0.034 * scale, 0.000],
            [0.106 * scale, 0.052 * scale, 0.000],
            [0.122 * scale, 0.062 * scale, 0.003 * scale],
        ],
        linkOrientations=[
            [0, 0, 0, 1], [0, 0, 0, 1],
            [0, 0, 0, 1], [0, 0, 0, 1],
            p.getQuaternionFromEuler([np.pi / 2, 0, 0]),
        ],
        linkInertialFramePositions=[[0, 0, 0]] * 5,
        linkInertialFrameOrientations=[[0, 0, 0, 1]] * 5,
        linkParentIndices=[0, 1, 2, 3, 4],
        linkJointTypes=[p.JOINT_FIXED] * 5,
        linkJointAxis=[[0, 0, 1]] * 5,
    )
def spawn_fruit(kind, pos):
    if kind == "orange": return spawn_orange(pos)
    if kind == "pear":   return spawn_pear(pos)
    if kind == "banana": return spawn_banana(pos)
    raise ValueError(f"Unknown fruit: {kind}")
# ============================================================
# Distractors
# ============================================================
def spawn_distractor(pos):
    kind = random.choice(["box", "cylinder", "sphere", "elongated"])
    color_pool = [
        [0.15, 0.40, 1.0, 1.0], [0.80, 0.10, 0.20, 1.0],
        [0.80, 0.80, 0.85, 1.0], [0.55, 0.20, 0.75, 1.0],
        [0.95, 0.90, 0.25, 1.0], [0.35, 0.80, 0.20, 1.0],
    ]
    color = random.choice(color_pool)
    if kind == "box":
        hx = float(np.random.uniform(0.018, 0.036))
        hy = float(np.random.uniform(0.018, 0.036))
        hz = float(np.random.uniform(0.015, 0.030))
        col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[hx, hy, hz])
        vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[hx, hy, hz],
                                  rgbaColor=color)
    elif kind == "cylinder":
        r   = float(np.random.uniform(0.015, 0.028))
        h   = float(np.random.uniform(0.025, 0.055))
        col = p.createCollisionShape(p.GEOM_CYLINDER, radius=r, height=h)
        vis = p.createVisualShape(p.GEOM_CYLINDER, radius=r, length=h,
                                  rgbaColor=color)
    elif kind == "elongated":
        hx = float(np.random.uniform(0.030, 0.055))
        hy = float(np.random.uniform(0.010, 0.018))
        hz = float(np.random.uniform(0.010, 0.018))
        col = p.createCollisionShape(p.GEOM_BOX, halfExtents=[hx, hy, hz])
        vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[hx, hy, hz],
                                  rgbaColor=color)
    else:
        r   = float(np.random.uniform(0.018, 0.030))
        col = p.createCollisionShape(p.GEOM_SPHERE, radius=r)
        vis = p.createVisualShape(p.GEOM_SPHERE, radius=r, rgbaColor=color)
    return p.createMultiBody(
        baseMass=0.03,
        baseCollisionShapeIndex=col,
        baseVisualShapeIndex=vis,
        basePosition=pos,
        baseOrientation=p.getQuaternionFromEuler(
            [0, 0, np.random.uniform(-np.pi, np.pi)]),
    )
# ============================================================
# Photometric augmentation
# ============================================================
def apply_photometric_augmentation(rgb):
    img = rgb.copy()
    if random.random() < 0.85:
        alpha = np.random.uniform(0.88, 1.15)
        beta  = np.random.uniform(-16, 16)
        img   = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)
    if random.random() < 0.75:
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[..., 0]  = (hsv[..., 0] + np.random.uniform(-5, 5)) % 180
        hsv[..., 1] *= np.random.uniform(0.82, 1.18)
        hsv[..., 2] *= np.random.uniform(0.85, 1.15)
        hsv[..., 1]  = np.clip(hsv[..., 1], 0, 255)
        hsv[..., 2]  = np.clip(hsv[..., 2], 0, 255)
        img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
    if random.random() < 0.20:
        k   = random.choice([3, 5])
        img = cv2.GaussianBlur(img, (k, k),
                               sigmaX=np.random.uniform(0.4, 1.1))
    if random.random() < 0.20:
        noise = np.random.normal(0, np.random.uniform(2, 8),
                                 img.shape).astype(np.float32)
        img   = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    if random.random() < 0.15:
        q   = int(np.random.uniform(55, 95))
        bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        ok, enc = cv2.imencode(".jpg", bgr, [int(cv2.IMWRITE_JPEG_QUALITY), q])
        if ok:
            bgr = cv2.imdecode(enc, cv2.IMREAD_COLOR)
            img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return img
# ============================================================
# Sampling plans
# ============================================================
def sample_balanced_fruit_plan(n, class_hist):
    plan       = []
    local_hist = Counter(class_hist)
    min_count  = min(local_hist.get(c, 0) for c in CFG.fruit_classes)
    least_seen = [c for c in CFG.fruit_classes
                  if local_hist.get(c, 0) == min_count]
    forced = random.choice(least_seen)
    plan.append(forced)
    local_hist[forced] += 1
    for _ in range(n - 1):
        min_count  = min(local_hist.get(c, 0) for c in CFG.fruit_classes)
        least_seen = [c for c in CFG.fruit_classes
                      if local_hist.get(c, 0) == min_count]
        cls = random.choice(least_seen)
        plan.append(cls)
        local_hist[cls] += 1
    random.shuffle(plan)
    return plan
def sample_target_focused_fruit_plan(n, class_hist, target_class):
    plan         = []
    local_hist   = Counter(class_hist)
    target_count = min(n, random.choice([2, 2, 3]))
    for _ in range(target_count):
        plan.append(target_class)
        local_hist[target_class] += 1
    while len(plan) < n:
        min_count  = min(local_hist.get(c, 0) for c in CFG.fruit_classes)
        least_seen = [c for c in CFG.fruit_classes
                      if local_hist.get(c, 0) == min_count]
        cls = random.choice(least_seen)
        plan.append(cls)
        local_hist[cls] += 1
    random.shuffle(plan)
    return plan
def needs_balanced_scene(image_class_hist):
    non_target = [c for c in CFG.fruit_classes if c != CFG.target_class]
    return any(image_class_hist.get(c, 0) < CFG.class_min_box_quota
               for c in non_target)
def choose_view_modes(split_name, count):
    if split_name == "train":
        base = ["eval", "eval", "eval", "eval_jitter"]
    else:
        base = ["eval", "eval", "eval_jitter"]
    return [base[i % len(base)] for i in range(count)]
# ============================================================
# Label extraction
# ============================================================
def collect_gt_boxes(fruit_bodies, seg, img_w, img_h):
    yolo_lines    = []
    gt_boxes      = []
    class_counter = Counter()
    dropped_mask  = 0
    dropped_size  = 0
    for bid, cls in fruit_bodies.items():
        bb, mask_pixels = bbox_from_mask_union(seg, bid)
        if mask_pixels < CFG.min_mask_pixels:
            dropped_mask += 1
            continue
        if bb is None:
            continue
        bb = pad_bbox(bb, CFG.bbox_pad, img_w, img_h)
        if bb is None:
            continue
        x1, y1, x2, y2 = bb
        bw   = x2 - x1 + 1
        bh   = y2 - y1 + 1
        area = bw * bh
        if area < CFG.min_area or min(bw, bh) < CFG.min_box_side:
            dropped_size += 1
            continue
        yolo_lines.append(xyxy_to_yolo_line(bb, CLASS_TO_ID[cls],
                                            img_w, img_h))
        gt_boxes.append((bb, cls))
        class_counter[cls] += 1
    return yolo_lines, gt_boxes, class_counter, dropped_mask, dropped_size
# ============================================================
# Dataset generation — unchanged from v11
# ============================================================
def generate_dataset(
    outdir=CFG.outdir,
    train_episodes=CFG.train_episodes,
    val_episodes=CFG.val_episodes,
    save_samples=CFG.save_samples,
    samples_dir=CFG.samples_dir,
    seed=CFG.seed,
    background_keep_prob=0.05,
):
    seed_everything(seed)
    reset_dir(outdir)
    ensure_sample_dirs(samples_dir)
    for split in ["train", "val"]:
        ensure_dir(os.path.join(outdir, "images", split))
        ensure_dir(os.path.join(outdir, "labels", split))
    write_dataset_yaml(outdir, CFG.fruit_classes)
    connect_direct()
    class_hist        = Counter()
    sample_save_count = 0
    def run_split(split_name, episodes, views_per_scene,
                  empty_scene_prob, target_focus_prob):
        nonlocal sample_save_count
        nonempty_images    = 0
        skipped_background = 0
        total_boxes        = 0
        image_class_hist   = Counter()
        total_drop_mask    = 0
        total_drop_size    = 0
        for ep in trange(episodes, desc=f"Generating {split_name}"):
            reset_world()
            _, _, table_top_z = load_scene()
            fruit_bodies  = {}
            n_distractors = np.random.randint(0,
                                              CFG.distractors_per_scene_max + 1)
            is_empty = (random.random() < empty_scene_prob)
            if is_empty:
                fruit_plan = []
            else:
                n_fruits = np.random.randint(CFG.fruits_per_scene_min,
                                             CFG.fruits_per_scene_max + 1)
                force_balanced = (split_name == "train" and
                                  needs_balanced_scene(image_class_hist))
                use_focused    = (not force_balanced and
                                  random.random() < target_focus_prob)
                if use_focused:
                    fruit_plan = sample_target_focused_fruit_plan(
                        n_fruits, class_hist, CFG.target_class)
                else:
                    fruit_plan = sample_balanced_fruit_plan(
                        n_fruits, class_hist)
            positions = sample_positions_for_objects(
                table_top_z, fruit_plan, n_distractors, min_dist=0.085)
            idx = 0
            for cls in fruit_plan:
                bid = spawn_fruit(cls, positions[idx])
                fruit_bodies[bid] = cls
                class_hist[cls]   += 1
                idx += 1
            for _ in range(n_distractors):
                spawn_distractor(positions[idx])
                idx += 1
            for _ in range(np.random.randint(80, 130)):
                p.stepSimulation()
            view_modes = choose_view_modes(split_name, views_per_scene)
            for view_idx, mode in enumerate(view_modes):
                rgb, seg = render_camera(CFG.cam_w, CFG.cam_h, mode=mode)
                yolo_lines, gt_boxes, found_hist, d_mask, d_size = \
                    collect_gt_boxes(fruit_bodies, seg, CFG.cam_w, CFG.cam_h)
                total_drop_mask += d_mask
                total_drop_size += d_size
                if len(yolo_lines) == 0:
                    if random.random() > background_keep_prob:
                        skipped_background += 1
                        continue
                rgb_aug  = apply_photometric_augmentation(rgb)
                stem     = f"{split_name}_{ep:06d}_v{view_idx:02d}"
                img_path = os.path.join(outdir, "images", split_name,
                                        stem + ".png")
                lbl_path = os.path.join(outdir, "labels", split_name,
                                        stem + ".txt")
                cv2.imwrite(img_path,
                            cv2.cvtColor(rgb_aug, cv2.COLOR_RGB2BGR))
                with open(lbl_path, "w") as f:
                    f.write("\n".join(yolo_lines))
                if len(yolo_lines) > 0:
                    nonempty_images += 1
                    total_boxes     += len(yolo_lines)
                    image_class_hist.update(found_hist)
                if save_samples > 0 and sample_save_count < save_samples:
                    vis = draw_bboxes(rgb_aug, gt_boxes,
                                      title=f"{split_name}:{ep}:v{view_idx}:{mode}")
                    cv2.imwrite(
                        os.path.join(samples_dir, "generate",
                                     f"generate_scene_{sample_save_count:04d}.png"),
                        vis)
                    sample_save_count += 1
        print(f"[{split_name}] nonempty images:    {nonempty_images}")
        print(f"[{split_name}] background skipped: {skipped_background}")
        print(f"[{split_name}] total boxes:        {total_boxes}")
        print(f"[{split_name}] boxes by class:     {dict(image_class_hist)}")
        print(f"[{split_name}] dropped mask:       {total_drop_mask}, "
              f"dropped size: {total_drop_size}")
    run_split("train", train_episodes, CFG.views_per_scene_train,
              CFG.empty_scene_prob_train, CFG.target_focused_scene_prob_train)
    run_split("val",   val_episodes,   CFG.views_per_scene_val,
              CFG.empty_scene_prob_val,  CFG.target_focused_scene_prob_val)
    p.disconnect()
    print("\n[DONE] Dataset generated")
    print(f"Outdir: {outdir}")
    print(f"Scene-level class histogram: {dict(class_hist)}")
    if (not glob(os.path.join(outdir, "images", "train", "*.png")) or
            not glob(os.path.join(outdir, "images", "val", "*.png"))):
        raise RuntimeError("Dataset generation failed: images not found.")
# ============================================================
# Training — unchanged from v11
# ============================================================
def train_detector(
    dataset_dir=CFG.outdir,
    model_path=CFG.model_path,
    classes_path=CFG.classes_path,
    epochs=CFG.epochs,
    imgsz=CFG.imgsz,
    batch_size=CFG.batch_size,
    pretrained=CFG.pretrained,
):
    yaml_path = os.path.join(dataset_dir, "data.yaml")
    if not os.path.exists(yaml_path):
        raise FileNotFoundError(f"Missing dataset yaml: {yaml_path}")
    print(f"[TRAIN] Starting on GPU: {torch.cuda.get_device_name(DEVICE)}")
    torch.cuda.empty_cache()
    model = YOLO(pretrained)
    model.train(
        data=yaml_path, epochs=epochs, imgsz=imgsz, batch=batch_size,
        device=DEVICE, project="runs", name="fruit_detector_v14",
        exist_ok=True, verbose=True, cache=True, patience=40,
        cos_lr=True, close_mosaic=5, degrees=6.0, translate=0.08,
        scale=0.20, shear=2.0, perspective=0.0, fliplr=0.5, flipud=0.0,
        hsv_h=0.012, hsv_s=0.40, hsv_v=0.25, mosaic=0.30, mixup=0.05,
        amp=True, plots=True, workers=2,
    )
    best_path = str(model.trainer.best)
    ensure_dir(os.path.dirname(model_path))
    shutil.copy(best_path, model_path)
    ensure_dir(os.path.dirname(classes_path))
    with open(classes_path, "w") as f:
        json.dump(list(CFG.fruit_classes), f)
    print(f"\n[DONE] Best model saved to: {model_path}")
    print(f"[DONE] Classes saved to:    {classes_path}")
    metrics = model.val(data=yaml_path, imgsz=imgsz, split="val",
                        device=DEVICE)
    print("[VAL] Validation completed")
    return metrics
# ============================================================
# Inference helpers
# ============================================================
def load_detector(model_path, classes_path):
    if not os.path.exists(model_path):
        raise FileNotFoundError(model_path)
    if not os.path.exists(classes_path):
        raise FileNotFoundError(classes_path)
    model = YOLO(model_path)
    with open(classes_path, "r") as f:
        class_names = json.load(f)
    return model, class_names
def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    iw     = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    ih     = max(0.0, min(ay2, by2) - max(ay1, by1))
    inter  = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    return inter / (area_a + area_b - inter + 1e-9)
def dedupe_detections(detections, iou_thresh=0.60):
    detections = sorted(detections, key=lambda x: x[0], reverse=True)
    kept = []
    for conf, cls_name, bb in detections:
        if not any(cls_name == k_cls and box_iou(bb, k_bb) >= iou_thresh
                   for _, k_cls, k_bb in kept):
            kept.append((conf, cls_name, bb))
    return kept
def predict_once(model, class_names, bgr, conf, imgsz, augment):
    result = model.predict(
        source=bgr, conf=conf, imgsz=imgsz, iou=0.45,
        augment=augment, device=DEVICE, verbose=False
    )[0]
    dets = []
    if result.boxes is None or len(result.boxes) == 0:
        return dets
    for bb, c, cls_id in zip(result.boxes.xyxy.cpu().numpy(),
                              result.boxes.conf.cpu().numpy(),
                              result.boxes.cls.cpu().numpy().astype(int)):
        x1, y1, x2, y2 = [int(v) for v in bb]
        dets.append((float(c), class_names[int(cls_id)], (x1, y1, x2, y2)))
    return dets
def detect_scene(model, class_names, rgb, conf_thresh=0.001, imgsz=800):
    bgr      = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    all_dets = []
    all_dets.extend(predict_once(model, class_names, bgr,
                                 conf_thresh, imgsz, augment=False))
    all_dets.extend(predict_once(model, class_names, bgr,
                                 max(0.0005, conf_thresh * 0.5),
                                 imgsz, augment=True))
    all_dets = dedupe_detections(all_dets, iou_thresh=0.60)
    all_dets.sort(key=lambda x: x[0], reverse=True)
    return all_dets
def match_detection_to_body(det_bb, gt_boxes_by_body, min_iou=0.01):
    best_body, best_iou = None, 0.0
    for body_id, gt_bb in gt_boxes_by_body.items():
        if gt_bb is None:
            continue
        iou = box_iou(det_bb, gt_bb)
        if iou > best_iou:
            best_iou  = iou
            best_body = body_id
    if best_body is None or best_iou < min_iou:
        return None, 0.0
    return best_body, best_iou
# ============================================================
# Franka control
# ============================================================
def find_panda_indices(robot_id):
    arm, fingers, ee_link = [], [], None
    for j in range(p.getNumJoints(robot_id)):
        info  = p.getJointInfo(robot_id, j)
        jname = info[1].decode("utf-8")
        lname = info[12].decode("utf-8")
        if jname.startswith("panda_joint"):
            arm.append(j)
        if jname in ["panda_finger_joint1", "panda_finger_joint2"]:
            fingers.append(j)
        if lname == "panda_hand":
            ee_link = j
    arm     = sorted(arm)[:7]
    fingers = sorted(fingers)
    if ee_link is None:
        ee_link = p.getNumJoints(robot_id) - 1
    return arm, fingers, ee_link
def find_link_index(robot_id, link_name):
    for j in range(p.getNumJoints(robot_id)):
        if p.getJointInfo(robot_id, j)[12].decode("utf-8") == link_name:
            return j
    return None
# ============================================================
# Grasp and lift
#
# grasp_mode="v13": the original procedure (kept for the before/after
#   comparison). Aims the panda_hand frame at table+0.06 m, which puts
#   the fingertips ~4 cm below the table surface, then attaches a fixed
#   constraint with both frames at the body origins, which snaps the
#   object into the hand.
#
# grasp_mode="v14":
#   - IK targets the panda_grasptarget link (between the fingertips)
#     at the object's own height, never below grasp_min_clearance.
#   - Gripper yaw is aligned to the object's yaw, so the fingers close
#     across the banana's long axis rather than along it.
#   - Motion is broken into waypoints (straight-line Cartesian moves).
#   - Both fingers must be touching the target before the grasp
#     counts (require_finger_contact); otherwise -> "grasp_missed".
#   - The constraint keeps the object's current in-hand pose instead
#     of snapping its origin to the hand origin.
# ============================================================
def _wrap_yaw(yaw):
    # parallel-jaw gripper is symmetric under 180 deg; keep within +-90 deg
    while yaw > np.pi / 2:
        yaw -= np.pi
    while yaw < -np.pi / 2:
        yaw += np.pi
    return yaw
def grasp_point_and_axis(obj_id):
    """Where to put the fingertips, and the object's long-axis yaw.
    Multi-link fruit: banana is a chain of 5 spheres + stem, pear is a
    base sphere + smaller top sphere + stem. Grasp a part that fits in
    the Panda's 8 cm opening:
      orange (1 link)  -> the sphere itself
      pear             -> the smaller top sphere (base can exceed 8 cm)
      banana           -> the middle sphere of the chain
    The axis yaw runs from the base toward the far end of the chain."""
    n = p.getNumJoints(obj_id)
    base_pos, base_orn = p.getBasePositionAndOrientation(obj_id)
    if n == 0:
        return list(base_pos), p.getEulerFromQuaternion(base_orn)[2]
    link_pos = [p.getLinkState(obj_id, j, computeForwardKinematics=True)[0]
                for j in range(n)]
    if n >= 5:                       # banana
        gp  = link_pos[CFG.banana_grasp_link]
        tip = link_pos[3]
    else:                            # pear: grasp the neck, just below the top sphere
        gp  = [link_pos[0][0], link_pos[0][1], link_pos[0][2] - CFG.pear_grasp_drop]
        tip = link_pos[0]
    yaw = float(np.arctan2(tip[1] - base_pos[1], tip[0] - base_pos[0]))
    if n < 5:                        # pear axis is vertical; use body yaw
        yaw = p.getEulerFromQuaternion(base_orn)[2]
    return list(gp), yaw
def attempt_grasp_and_lift(robot_id, arm_joints, finger_joints, ee_link,
                           obj_id, table_top_z, lift_threshold,
                           writer=None, vid_w=640, vid_h=480,
                           detections=None, grasp_mode=None):
    """
    Grasp and lift the object.
    Returns (success: bool, lift_height: float, reason: str), where
    reason is one of "success", "grasp_missed", "lift_insufficient".
    """
    grasp_mode = grasp_mode or CFG.grasp_mode
    obj_pos, obj_orn = p.getBasePositionAndOrientation(obj_id)
    # V14-1: video camera aimed at the table top (v13 aimed at z=0.10)
    T = table_top_z
    vid_cam = [1.30, -0.55, T + 0.55]
    vid_tgt = [0.45,  0.00, T + 0.12]
    vid_fov = 55.0
    if grasp_mode == "v13":
        ctrl_link = ee_link
        grasp_orn = p.getQuaternionFromEuler([np.pi, 0, 0])
        pre_grasp = [obj_pos[0], obj_pos[1], table_top_z + 0.35]
        grasp_pos = [obj_pos[0], obj_pos[1], table_top_z + 0.06]
        lift_pos  = [obj_pos[0], obj_pos[1],
                     table_top_z + lift_threshold + CFG.lift_above_threshold]
        n_wp = 1
    else:
        gt_link   = find_link_index(robot_id, "panda_grasptarget")
        ctrl_link = gt_link if gt_link is not None else ee_link
        gp, axis_yaw = grasp_point_and_axis(obj_id)
        grasp_orn = p.getQuaternionFromEuler(
            [np.pi, 0, _wrap_yaw(axis_yaw + CFG.grasp_yaw_offset)])
        grasp_z   = max(gp[2], table_top_z + CFG.grasp_min_clearance)
        pre_grasp = [gp[0], gp[1], grasp_z + CFG.pregrasp_clearance]
        grasp_pos = [gp[0], gp[1], grasp_z]
        lift_pos  = [obj_pos[0], obj_pos[1],
                     table_top_z + lift_threshold + CFG.lift_target_above]
        n_wp = CFG.move_waypoints
    def record(label):
        # Detection boxes are in eval-camera pixels, so they are NOT drawn on
        # the video camera's frames (v13 drew them in the wrong place); the
        # detection itself is shown on the opening frames written by run_trials.
        if writer is not None:
            writer.write(render_video_frame(vid_w, vid_h, vid_cam, vid_tgt,
                                            vid_fov, detections=None,
                                            label=label))
    def step_and_record(target, orn, steps, label, waypoints=1):
        start = np.array(p.getLinkState(robot_id, ctrl_link,
                                        computeForwardKinematics=True)[4])
        target = np.array(target)
        per_wp = max(1, steps // waypoints)
        k = 0
        for w in range(1, waypoints + 1):
            wp = start + (target - start) * (w / waypoints)
            q = p.calculateInverseKinematics(robot_id, ctrl_link, wp.tolist(),
                                             orn, maxNumIterations=200,
                                             residualThreshold=1e-4)
            q_arm = q[:len(arm_joints)]
            for _ in range(per_wp):
                for i, j in enumerate(arm_joints):
                    p.setJointMotorControl2(robot_id, j, p.POSITION_CONTROL,
                                            targetPosition=float(q_arm[i]),
                                            force=220)
                p.stepSimulation()
                if k % 3 == 0:
                    record(label)
                k += 1
    gear_cid = None
    if grasp_mode != "v13" and CFG.couple_fingers and len(finger_joints) == 2:
        # keep the two fingers symmetric (as in PyBullet's own Panda demo),
        # so one finger cannot push a round fruit away before the other arrives
        gear_cid = p.createConstraint(robot_id, finger_joints[0],
                                      robot_id, finger_joints[1],
                                      jointType=p.JOINT_GEAR, jointAxis=[1, 0, 0],
                                      parentFramePosition=[0, 0, 0],
                                      childFramePosition=[0, 0, 0])
        p.changeConstraint(gear_cid, gearRatio=-1, erp=0.1, maxForce=50)
    def grip_and_record(open_grip, steps, label):
        target = 0.04 if open_grip else 0.0
        vmax = CFG.finger_close_velocity if grasp_mode != "v13" else None
        for i_step in range(steps):
            for j in finger_joints:
                if vmax is None:
                    p.setJointMotorControl2(robot_id, j, p.POSITION_CONTROL,
                                            targetPosition=target, force=60)
                else:
                    p.setJointMotorControl2(robot_id, j, p.POSITION_CONTROL,
                                            targetPosition=target, force=60,
                                            maxVelocity=vmax)
            p.stepSimulation()
            if i_step % 3 == 0:
                record(label)
    def settle_and_record(steps, label):
        for i_step in range(steps):
            p.stepSimulation()
            if i_step % 3 == 0:
                record(label)
    # Phase 1: open gripper, approach
    grip_and_record(True, 80, "Opening gripper")
    step_and_record(pre_grasp, grasp_orn, 200, "Approaching target", n_wp)
    # Phase 2: descend
    step_and_record(grasp_pos, grasp_orn, 200, "Descending to target", n_wp)
    # Phase 3: IK settle
    settle_and_record(CFG.ik_settle_steps, "Stabilising at grasp pose")
    # Phase 4: close gripper, then attach constraint
    grip_and_record(False, CFG.grip_close_steps, "Closing gripper...")
    settle_and_record(CFG.post_grip_settle, "Settling before constraint...")
    if grasp_mode != "v13" and CFG.require_finger_contact:
        touching = [len(p.getContactPoints(robot_id, obj_id, linkIndexA=f)) > 0
                    for f in finger_joints]
        if not all(touching):
            pos, _ = p.getBasePositionAndOrientation(obj_id)
            if gear_cid is not None:
                p.removeConstraint(gear_cid)
            return False, pos[2] - table_top_z, "grasp_missed"
    if grasp_mode == "v13":
        cid = p.createConstraint(
            parentBodyUniqueId=robot_id, parentLinkIndex=ee_link,
            childBodyUniqueId=obj_id, childLinkIndex=-1,
            jointType=p.JOINT_FIXED, jointAxis=[0, 0, 0],
            parentFramePosition=[0, 0, CFG.constraint_parent_offset],
            childFramePosition=[0, 0, CFG.constraint_child_offset],
        )
    else:
        ls = p.getLinkState(robot_id, ctrl_link, computeForwardKinematics=True)
        link_pos, link_orn = ls[4], ls[5]
        o_pos, o_orn = p.getBasePositionAndOrientation(obj_id)
        inv_pos, inv_orn = p.invertTransform(link_pos, link_orn)
        rel_pos, rel_orn = p.multiplyTransforms(inv_pos, inv_orn, o_pos, o_orn)
        cid = p.createConstraint(
            parentBodyUniqueId=robot_id, parentLinkIndex=ctrl_link,
            childBodyUniqueId=obj_id, childLinkIndex=-1,
            jointType=p.JOINT_FIXED, jointAxis=[0, 0, 0],
            parentFramePosition=rel_pos, childFramePosition=[0, 0, 0],
            parentFrameOrientation=rel_orn,
            childFrameOrientation=[0, 0, 0, 1],
        )
        p.changeConstraint(cid, maxForce=CFG.constraint_max_force)
    settle_and_record(CFG.post_constraint_settle, "Constraint settling...")
    # Phase 5: lift
    step_and_record(lift_pos, grasp_orn, CFG.lift_steps, "Lifting target!", n_wp)
    for _ in range(CFG.post_lift_settle):
        p.stepSimulation()
    final_pos, _ = p.getBasePositionAndOrientation(obj_id)
    lift_height  = final_pos[2] - table_top_z
    success      = lift_height >= lift_threshold
    if success:
        for _ in range(CFG.video_fps):
            record("SUCCESS - target lifted!")
    p.removeConstraint(cid)
    if gear_cid is not None:
        p.removeConstraint(gear_cid)
    return success, lift_height, ("success" if success else "lift_insufficient")
# ============================================================
# Trial scene helpers (shared by run_trials, diagnose_trials, grasp test)
# ============================================================
TRIAL_OUTCOMES = ["success", "zone_skip", "target_off_table",
                  "target_not_visible", "no_detection", "bad_localization",
                  "wrong_class", "grasp_missed", "lift_insufficient"]
def build_trial_scene(requested, fruits_in_scene):
    """Reset the world, spawn a trial scene, and let it settle.
    Returns a dict with robot handles, fruit ids/classes and the target id."""
    reset_world()
    robot_id, _, table_top_z = load_scene()
    arm_joints, finger_joints, ee_link = find_panda_indices(robot_id)
    fruit_plan = [requested]
    for _ in range(max(0, fruits_in_scene - 1)):
        fruit_plan.append(random.choice(CFG.fruit_classes))
    spawned  = spawn_trial_scene(table_top_z, fruit_plan)
    fruit_gt = {fid: cls for fid, cls in spawned}
    for _ in range(CFG.trial_settle_steps):
        p.stepSimulation()
    return dict(robot_id=robot_id, arm=arm_joints, fingers=finger_joints,
                ee=ee_link, table_top_z=table_top_z, fruit_gt=fruit_gt,
                fruit_ids=[fid for fid, _ in spawned],
                target_fid=spawned[0][0],
                has_second_same_class=any(c == requested
                                          for _, c in spawned[1:]))
def pre_detection_gate(scene):
    """Scene-level checks that happen before the detector runs.
    Returns (outcome or None, rgb, seg, gt_boxes_by_body, target_pixels)."""
    tfid, top = scene["target_fid"], scene["table_top_z"]
    pos, _ = p.getBasePositionAndOrientation(tfid)
    if pos[2] < top - 0.02:
        return "target_off_table", None, None, {}, 0
    if CFG.use_zone_check and not target_in_check_zone(tfid):
        return "zone_skip", None, None, {}, 0
    rgb, seg = render_camera(CFG.cam_w, CFG.cam_h, mode="eval")
    gt_boxes_by_body = {}
    for fid in scene["fruit_ids"]:
        bb, _ = bbox_from_mask_union(seg, fid)
        gt_boxes_by_body[fid] = bb
    target_pixels = int(seg_mask_for_body_all_links(seg, tfid).sum())
    if target_pixels < CFG.min_visible_pixels:
        return "target_not_visible", rgb, seg, gt_boxes_by_body, target_pixels
    return None, rgb, seg, gt_boxes_by_body, target_pixels
def save_results(name, summary, rows):
    ensure_dir(CFG.results_dir)
    with open(os.path.join(CFG.results_dir, f"{name}.json"), "w") as f:
        json.dump({"summary": summary, "rows": rows}, f, indent=2, default=str)
    print(f"[SAVED] {os.path.join(CFG.results_dir, name + '.json')}")
# ============================================================
# End-to-end evaluation
# ============================================================
def run_trials(
    requested=CFG.requested,
    model_path=CFG.model_path,
    classes_path=CFG.classes_path,
    trials=CFG.trials,
    fruits_in_scene=CFG.fruits_in_scene,
    lift_height_threshold=CFG.lift_height_threshold,
    save_samples=CFG.save_samples,
    samples_dir=CFG.samples_dir,
    conf_thresh=CFG.conf_thresh,
    match_iou_thresh=CFG.trial_match_iou_thresh,
    video_path=None,
    video_fps=CFG.video_fps,
    video_cam_w=CFG.video_cam_w,
    video_cam_h=CFG.video_cam_h,
    seed=CFG.trial_seed,
    grasp_mode=None,
    tag=None,
    model=None,
    class_names=None,
):
    """Every trial ends in exactly one outcome from TRIAL_OUTCOMES."""
    if requested not in CFG.fruit_classes:
        raise ValueError(f"requested must be one of {CFG.fruit_classes}")
    grasp_mode = grasp_mode or CFG.grasp_mode
    tag        = tag or f"trials_{requested}_{grasp_mode}_seed{seed}"
    video_path = video_path or f"successful_grasp_{requested}.mp4"
    if model is None:
        model, class_names = load_detector(model_path, classes_path)
    if seed is not None:
        seed_everything(seed)
    run_dir = os.path.join(samples_dir, "run", tag)
    ensure_dir(run_dir)
    connect_direct()
    print(f"[RUN] {tag}: requested={requested}, trials={trials}, "
          f"grasp_mode={grasp_mode}, seed={seed}, conf={conf_thresh}, "
          f"match IoU>={match_iou_thresh}, success: lift>={lift_height_threshold:.2f} m")
    counts, rows = Counter(), []
    saved_count, video_recorded = 0, False
    for t in range(1, trials + 1):
        scene = build_trial_scene(requested, fruits_in_scene)
        fruit_gt, tfid = scene["fruit_gt"], scene["target_fid"]
        row = dict(trial=t, outcome=None, conf=None, match_iou=None,
                   matched_target_instance=None, lift_height=None,
                   target_pixels=None,
                   has_second_same_class=scene["has_second_same_class"])
        outcome, rgb, seg, gt_boxes_by_body, tpx = pre_detection_gate(scene)
        row["target_pixels"] = tpx
        detections = []
        if outcome is None:
            detections = detect_scene(model, class_names, rgb,
                                      conf_thresh=conf_thresh, imgsz=CFG.imgsz)
        if rgb is not None and save_samples > 0 and saved_count < save_samples:
            gt_vis = [(bb, fruit_gt[fid]) for fid, bb in gt_boxes_by_body.items()
                      if bb is not None]
            cv2.imwrite(os.path.join(run_dir, f"trial_{t:02d}_gt.png"),
                        draw_bboxes(rgb, gt_vis, title=f"GT trial {t}"))
            cv2.imwrite(os.path.join(run_dir, f"trial_{t:02d}_pred.png"),
                        draw_bboxes(rgb, [(bb, f"{c}:{cf:.2f}") for cf, c, bb in detections],
                                    title=f"PRED trial {t}"))
            saved_count += 1
        target = None
        if outcome is None:
            req_dets = [d for d in detections if d[1] == requested]
            for conf, _, det_bb in req_dets:
                body, miou = match_detection_to_body(det_bb, gt_boxes_by_body,
                                                     min_iou=match_iou_thresh)
                if body is not None:
                    target = (body, conf, det_bb, miou)
                    break
            if target is None:
                outcome = "bad_localization" if req_dets else "no_detection"
        if outcome is None:
            obj_id, conf, det_bb, miou = target
            row.update(conf=round(conf, 4), match_iou=round(miou, 4),
                       matched_target_instance=(obj_id == tfid))
            if fruit_gt[obj_id] != requested:
                outcome = "wrong_class"
        if outcome is None:
            writer = None
            if not video_recorded:
                writer = open_video_writer(video_path, video_cam_w,
                                           video_cam_h, video_fps)
                det_frame = draw_bboxes(rgb, [(det_bb, requested)],
                                        title=f"Detected: {requested} (conf={conf:.2f})")
                for _ in range(video_fps):
                    writer.write(det_frame)
            ok, lift_h, outcome = attempt_grasp_and_lift(
                scene["robot_id"], scene["arm"], scene["fingers"], scene["ee"],
                obj_id, scene["table_top_z"], lift_height_threshold,
                writer=writer, vid_w=video_cam_w, vid_h=video_cam_h,
                detections=[(conf, requested, det_bb)], grasp_mode=grasp_mode)
            row["lift_height"] = round(lift_h, 4)
            if writer is not None:
                writer.release()
                if ok:
                    video_recorded = True
                elif os.path.exists(video_path):
                    os.remove(video_path)
        row["outcome"] = outcome
        counts[outcome] += 1
        rows.append(row)
        print(f"Trial {t:02d}: {outcome:18s} px={tpx:>5}  "
              f"conf={row['conf']}  lift={row['lift_height']}")
    p.disconnect()
    reached_detector = trials - sum(counts[k] for k in
                                    ["zone_skip", "target_off_table", "target_not_visible"])
    matched   = sum(counts[k] for k in ["wrong_class", "grasp_missed",
                                        "lift_insufficient", "success"])
    attempts  = sum(counts[k] for k in ["grasp_missed", "lift_insufficient", "success"])
    summary = dict(tag=tag, requested=requested, trials=trials, seed=seed,
                   grasp_mode=grasp_mode,
                   counts={k: counts.get(k, 0) for k in TRIAL_OUTCOMES},
                   reached_detector=reached_detector,
                   matched_detections=matched,
                   trial_detection_recall=(matched / reached_detector
                                           if reached_detector else None),
                   grasp_attempts=attempts,
                   grasp_success_per_attempt=(counts["success"] / attempts
                                              if attempts else None),
                   end_to_end_success=counts["success"] / trials,
                   video=video_path if video_recorded else None)
    print("\n===== End-to-End Results:", tag, "=====")
    for k in TRIAL_OUTCOMES:
        print(f"  {k:20s}: {counts.get(k, 0)}")
    print(f"  visible targets reaching detector: {reached_detector}")
    print(f"  matched detections:                {matched}"
          + (f"  (recall {matched / reached_detector:.1%})" if reached_detector else ""))
    print(f"  grasp success per attempt:         {counts['success']}/{attempts}")
    print(f"  end-to-end success:                {counts['success']}/{trials}")
    save_results(tag, summary, rows)
    return summary, rows
# ============================================================
# Grasp-only evaluation (no detector): isolates grasp mechanics
# ============================================================
def grasp_only_test(requested="banana", trials=50, grasp_mode=None,
                    seed=CFG.grasp_test_seed,
                    lift_height_threshold=CFG.lift_height_threshold):
    grasp_mode = grasp_mode or CFG.grasp_mode
    placement = "robot_on_table" if CFG.robot_on_table else "robot_on_floor"
    tag = f"grasp_only_{requested}_{grasp_mode}_{placement}_seed{seed}"
    seed_everything(seed)
    connect_direct()
    counts, rows = Counter(), []
    for t in range(1, trials + 1):
        scene = build_trial_scene(requested, CFG.fruits_in_scene)
        pos, _ = p.getBasePositionAndOrientation(scene["target_fid"])
        if pos[2] < scene["table_top_z"] - 0.02:
            outcome, h = "target_off_table", None
        else:
            _, h, outcome = attempt_grasp_and_lift(
                scene["robot_id"], scene["arm"], scene["fingers"], scene["ee"],
                scene["target_fid"], scene["table_top_z"],
                lift_height_threshold, grasp_mode=grasp_mode)
            h = round(h, 4)
        counts[outcome] += 1
        rows.append(dict(trial=t, outcome=outcome, lift_height=h))
    p.disconnect()
    attempts = sum(counts[k] for k in ["grasp_missed", "lift_insufficient", "success"])
    summary = dict(tag=tag, requested=requested, grasp_mode=grasp_mode,
                   robot_placement=placement,
                   seed=seed, counts=dict(counts), attempts=attempts,
                   success_per_attempt=counts["success"] / attempts if attempts else None)
    print(f"[GRASP-ONLY] {tag}: {counts['success']}/{attempts} successful "
          f"per attempt; {dict(counts)}")
    save_results(tag, summary, rows)
    return summary, rows
# ============================================================
# Display video in Colab
# ============================================================
def show_video_in_colab(video_path, caption=None):
    if not video_path or not os.path.exists(video_path):
        print(f"Video not found at {video_path}")
        return
    h264_path = video_path.replace(".mp4", "_h264.mp4")
    os.system(f"ffmpeg -y -i {video_path} "
              f"-vcodec libx264 -crf 23 -preset fast "
              f"{h264_path} -loglevel quiet")
    from IPython.display import HTML, display
    from base64 import b64encode
    src_path = h264_path if os.path.exists(h264_path) else video_path
    with open(src_path, "rb") as f:
        video_data = b64encode(f.read()).decode("utf-8")
    caption = caption or "Successful grasp trial"
    display(HTML(f"""
<video width="640" height="480" controls autoplay loop>
  <source src="data:video/mp4;base64,{video_data}" type="video/mp4">
</video>
<p><b>{caption}</b></p>
"""))
    print(f"Video file: {src_path}")
# ============================================================
# Main (command-line style entry point; the run section at the bottom
# of this file calls the same functions step by step instead)
# ============================================================
def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", choices=["generate", "train", "run", "all"],
                    default="all")
    ap.add_argument("--outdir",           default=CFG.outdir)
    ap.add_argument("--model_path",       default=CFG.model_path)
    ap.add_argument("--classes_path",     default=CFG.classes_path)
    ap.add_argument("--samples_dir",      default=CFG.samples_dir)
    ap.add_argument("--seed",             type=int,   default=CFG.seed)
    ap.add_argument("--train_episodes",   type=int,   default=CFG.train_episodes)
    ap.add_argument("--val_episodes",     type=int,   default=CFG.val_episodes)
    ap.add_argument("--epochs",           type=int,   default=CFG.epochs)
    ap.add_argument("--imgsz",            type=int,   default=CFG.imgsz)
    ap.add_argument("--batch_size",       type=int,   default=CFG.batch_size)
    ap.add_argument("--pretrained",       default=CFG.pretrained)
    ap.add_argument("--requested",
                    choices=list(CFG.fruit_classes), default=CFG.requested)
    ap.add_argument("--trials",           type=int,   default=CFG.trials)
    ap.add_argument("--conf_thresh",      type=float, default=CFG.conf_thresh)
    args = ap.parse_args([] if argv is None else argv)
    if args.mode in ["generate", "all"]:
        generate_dataset(
            outdir=args.outdir, train_episodes=args.train_episodes,
            val_episodes=args.val_episodes, samples_dir=args.samples_dir,
            seed=args.seed,
        )
    if args.mode in ["train", "all"]:
        train_detector(
            dataset_dir=args.outdir, model_path=args.model_path,
            classes_path=args.classes_path, epochs=args.epochs,
            imgsz=args.imgsz, batch_size=args.batch_size,
            pretrained=args.pretrained,
        )
    if args.mode in ["run", "all"]:
        summary, _ = run_trials(requested=args.requested,
                                model_path=args.model_path,
                                classes_path=args.classes_path,
                                trials=args.trials,
                                conf_thresh=args.conf_thresh)
        show_video_in_colab(summary["video"])

# ================================================================
# Detection-only diagnostic (no grasp): separates scene problems from
# detector problems, and records which class the detector assigns to the
# target, so class confusion is measured directly.
#
# Outcomes per trial:
#   target_off_table / target_not_visible : scene problem, not the CNN
#   true_miss            : no requested-class box even at conf 0.001
#   correct_above_thresh : requested-class box on the target, conf >= conf_thresh
#   correct_below_thresh : right box, confidence below conf_thresh
#   wrong_instance       : box on a different fruit of the SAME class
#   wrong_object         : box on a fruit of a different class
#   spatial_mismatch     : box overlaps no fruit at all
# ================================================================
PROBE_CONF_THRESH = 0.001
def diagnose_trials(requested=CFG.requested, trials=50,
                    fruits_in_scene=CFG.fruits_in_scene,
                    conf_thresh=CFG.conf_thresh,
                    match_iou_thresh=CFG.trial_match_iou_thresh,
                    samples_dir=CFG.samples_dir, max_saved=30,
                    seed=CFG.diag_seed, model=None, class_names=None):
    if model is None:
        model, class_names = load_detector(CFG.model_path, CFG.classes_path)
    seed_everything(seed)
    tag = f"diagnostic_{requested}_seed{seed}"
    diag_dir = os.path.join(samples_dir, "diagnostic", tag)
    ensure_dir(diag_dir)
    connect_direct()
    counts, rows, saved = Counter(), [], 0
    for t in range(1, trials + 1):
        scene = build_trial_scene(requested, fruits_in_scene)
        tfid, fruit_gt = scene["target_fid"], scene["fruit_gt"]
        outcome, rgb, seg, gtb, tpx = pre_detection_gate(scene)
        row = dict(trial=t, outcome=outcome, target_pixels=tpx,
                   has_second_same_class=scene["has_second_same_class"],
                   target_top_pred=None, target_top_conf=None)
        if outcome is None:
            tbox = gtb[tfid]
            dets_probe = detect_scene(model, class_names, rgb,
                                      conf_thresh=PROBE_CONF_THRESH, imgsz=CFG.imgsz)
            # what does the detector call the target? (any class, IoU >= 0.5)
            on_target = [d for d in dets_probe if box_iou(d[2], tbox) >= 0.5]
            if on_target:
                top = max(on_target, key=lambda d: d[0])
                row["target_top_pred"], row["target_top_conf"] = top[1], round(top[0], 4)
            req = [d for d in dets_probe if d[1] == requested]
            if not req:
                outcome = "true_miss"
            else:
                hits_target = [d for d in req if box_iou(d[2], tbox) >= match_iou_thresh]
                def hit_class(cls_filter):
                    return [d for d in req if any(
                        fid != tfid and bb is not None and cls_filter(fruit_gt[fid])
                        and box_iou(d[2], bb) >= match_iou_thresh
                        for fid, bb in gtb.items())]
                if any(d[0] >= conf_thresh for d in hits_target):
                    outcome = "correct_above_thresh"
                elif hits_target:
                    outcome = "correct_below_thresh"
                elif hit_class(lambda c: c == requested):
                    outcome = "wrong_instance"
                elif hit_class(lambda c: c != requested):
                    outcome = "wrong_object"
                else:
                    outcome = "spatial_mismatch"
            if outcome != "correct_above_thresh" and saved < max_saved:
                gt_vis = [(bb, fruit_gt[fid]) for fid, bb in gtb.items() if bb is not None]
                pred_vis = [(bb, f"{c}:{cf:.2f}") for cf, c, bb in dets_probe]
                cv2.imwrite(os.path.join(diag_dir, f"trial_{t:02d}_{outcome}.png"),
                            draw_bboxes(rgb, gt_vis + pred_vis, title=f"trial{t:02d} [{outcome}]"))
                saved += 1
        row["outcome"] = outcome
        counts[outcome] += 1
        rows.append(row)
        print(f"Trial {t:02d}: {outcome:22s} px={tpx:>5} "
              f"target_pred={row['target_top_pred']}:{row['target_top_conf']}")
    p.disconnect()
    keys = ["target_off_table", "target_not_visible", "true_miss", "spatial_mismatch",
            "wrong_object", "wrong_instance", "correct_below_thresh", "correct_above_thresh"]
    visible = trials - counts["target_off_table"] - counts["target_not_visible"]
    found = counts["correct_above_thresh"]
    preds = Counter(r["target_top_pred"] for r in rows if r["outcome"] not in
                    ("target_off_table", "target_not_visible"))
    print(f"\n===== Diagnostic breakdown: {tag} =====")
    for k in keys:
        print(f"{k:22s}: {counts.get(k, 0):3d}")
    print(f"Recall on visible targets: {found}/{visible}"
          + (f" ({found / visible:.1%})" if visible else ""))
    print(f"Detector's top label on the visible target: {dict(preds)}")
    summary = dict(tag=tag, requested=requested, seed=seed, trials=trials,
                   counts={k: counts.get(k, 0) for k in keys},
                   visible_targets=visible, found_above_thresh=found,
                   recall_visible=(found / visible if visible else None),
                   target_top_pred_counts=dict(preds))
    save_results(tag, summary, rows)
    return summary, rows

# ================================================================
# RUN
# ================================================================
if RUN_SANITY_CHECK:
    # ================================================================
    # Camera and scene sanity check (runs before dataset generation).
    # Confirms (1) the eval camera sees the trial target, (2) the robot is
    # not intersecting the table, and shows one frame from each camera.
    # ================================================================
    import matplotlib.pyplot as plt
    seed_everything(123)
    connect_direct()
    visible, gate_counts = 0, Counter()
    for t in range(20):
        scene = build_trial_scene(CFG.requested, CFG.fruits_in_scene)
        outcome, rgb, seg, gtb, px = pre_detection_gate(scene)
        gate_counts[outcome or "visible"] += 1
        if t == 0:
            eval_img = draw_bboxes(rgb, [(bb, scene["fruit_gt"][fid])
                                         for fid, bb in gtb.items() if bb is not None],
                                   title="eval camera + GT boxes")
            jit_rgb, _ = render_camera(CFG.cam_w, CFG.cam_h, mode="eval_jitter")
            T = scene["table_top_z"]
            vid_img = render_video_frame(640, 480, [1.30, -0.55, T + 0.55],
                                         [0.45, 0.00, T + 0.12], 55.0)
            robot_table_contacts = len(p.getContactPoints(scene["robot_id"]))
    p.disconnect()
    print("Trial-scene gate outcomes over 20 scenes:", dict(gate_counts))
    print("Robot contact points with other bodies at start:", robot_table_contacts)
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
    for a, img, title in zip(ax, [cv2.cvtColor(eval_img, cv2.COLOR_BGR2RGB), jit_rgb,
                                  cv2.cvtColor(vid_img, cv2.COLOR_BGR2RGB)],
                             ["eval (trial) camera", "eval_jitter (training)", "video camera"]):
        a.imshow(img); a.set_title(title); a.axis("off")
    plt.show()
    assert gate_counts["visible"] >= 18, "Camera is not seeing the trial target; do not generate data yet."
    assert robot_table_contacts == 0, "Robot is intersecting the table."
    print("Sanity check passed.")

LAST_CKPT = "runs/fruit_detector_v14/weights/last.pt"
if REUSE_TRAINED_MODEL and os.path.exists(CFG.model_path):
    print(f"[SKIP] Found trained model at {CFG.model_path}; skipping "
          f"dataset generation and training.")
else:
    # Dataset: 800 train + 150 val episodes, seed 42 (deterministic, so a
    # regenerated dataset is identical). Label counts per class print at
    # the end (Table II).
    if not os.path.exists(os.path.join(CFG.outdir, "data.yaml")):
        generate_dataset()
    # Training: YOLOv8m, 150 epochs, imgsz 800, batch 12 (unchanged from
    # v13). Per-class P / R / mAP50 / mAP50-95 print at the end (Table III);
    # confusion_matrix.png and PR curves go to runs/fruit_detector_v14/.
    if RESUME_TRAINING and os.path.exists(LAST_CKPT):
        # Colab disconnected mid-training: continue from the last epoch.
        print(f"[RESUME] Resuming training from {LAST_CKPT}")
        resumed = YOLO(LAST_CKPT)
        resumed.train(resume=True)
        ensure_dir(os.path.dirname(CFG.model_path))
        shutil.copy(str(resumed.trainer.best), CFG.model_path)
        with open(CFG.classes_path, "w") as f:
            json.dump(list(CFG.fruit_classes), f)
        val_metrics = YOLO(CFG.model_path).val(
            data=os.path.join(CFG.outdir, "data.yaml"), imgsz=CFG.imgsz,
            split="val", device=DEVICE)
    else:
        val_metrics = train_detector()

if RUN_TRIALS:
    # ================================================================
    # End-to-end trials: detect -> match -> grasp -> lift, 50 trials per
    # class, each class with its own fixed seed. Results: results_v14/*.json
    # ================================================================
    model, class_names = load_detector(CFG.model_path, CFG.classes_path)
    trial_summaries = {}
    for i, cls in enumerate(CFG.fruit_classes):
        summary, _ = run_trials(requested=cls, trials=50,
                                seed=CFG.trial_seed + i,
                                model=model, class_names=class_names)
        trial_summaries[cls] = summary
    show_video_in_colab(trial_summaries["banana"]["video"],
                        caption="Successful banana grasp trial (v14)")

if RUN_DIAGNOSTIC:
    model, class_names = load_detector(CFG.model_path, CFG.classes_path)
    diag_summaries = {}
    for i, cls in enumerate(CFG.fruit_classes):
        diag_summaries[cls], _ = diagnose_trials(requested=cls, trials=50,
                                                 seed=CFG.diag_seed + i,
                                                 model=model, class_names=class_names)

if RUN_GRASP_ONLY:
    # ================================================================
    # Grasp-only comparison (no detector; target position from the simulator).
    # Isolates grasp mechanics for the before/after table:
    #   A: v13 grasp, robot on the floor inside the table (original setup)
    #   B: v13 grasp, robot mounted on the table          (placement fix only)
    #   C: v14 grasp, robot mounted on the table, per class
    # Note: v13's grasp attaches the constraint whether or not the fingers
    # touch the fruit; v14 requires both fingers in contact. B therefore
    # overstates what a real gripper would achieve.
    # ================================================================
    grasp_summaries = {}
    CFG.robot_on_table = False
    grasp_summaries["A_v13_robot_on_floor"], _ = grasp_only_test("banana", 50, grasp_mode="v13")
    CFG.robot_on_table = True
    grasp_summaries["B_v13_robot_on_table"], _ = grasp_only_test("banana", 50, grasp_mode="v13")
    for cls in CFG.fruit_classes:
        grasp_summaries[f"C_v14_{cls}"], _ = grasp_only_test(cls, 50, grasp_mode="v14")

if RUN_SUMMARY:
    # ================================================================
    # Results summary: collects every JSON in results_v14/ into tables
    # (also saved as CSV) for the report.
    # ================================================================
    import pandas as pd
    from glob import glob
    recs = [json.load(open(f))["summary"] for f in sorted(glob(os.path.join(CFG.results_dir, "*.json")))]
    trial_rows, diag_rows, grasp_rows = [], [], []
    for s in recs:
        if s["tag"].startswith("trials_"):
            trial_rows.append({"class": s["requested"], **s["counts"],
                               "visible_reached_detector": s["reached_detector"],
                               "matched": s["matched_detections"],
                               "trial_recall": s["trial_detection_recall"],
                               "grasp_attempts": s["grasp_attempts"],
                               "grasp_success_per_attempt": s["grasp_success_per_attempt"],
                               "end_to_end_success": s["end_to_end_success"]})
        elif s["tag"].startswith("diagnostic_"):
            diag_rows.append({"class": s["requested"], **s["counts"],
                              "visible": s["visible_targets"],
                              "recall_visible": s["recall_visible"],
                              "target_top_pred": s["target_top_pred_counts"]})
        elif s["tag"].startswith("grasp_only_"):
            grasp_rows.append({"run": s["tag"], "mode": s["grasp_mode"],
                               "robot": s.get("robot_placement"),
                               "class": s["requested"], **s["counts"],
                               "attempts": s["attempts"],
                               "success_per_attempt": s["success_per_attempt"]})
    pd.set_option("display.width", 200)
    for name, rows in [("end_to_end_trials", trial_rows), ("diagnostic", diag_rows),
                       ("grasp_only", grasp_rows)]:
        if rows:
            df = pd.DataFrame(rows).fillna(0)
            df.to_csv(os.path.join(CFG.results_dir, f"summary_{name}.csv"), index=False)
            print(f"\n=== {name} ===")
            display(df)

print("\nAll done. Use File -> Save so the notebook keeps these outputs.")

# ================================================================
# DEPTH-CAMERA LOCALIZATION
#
# What changes: in v14 the arm is aimed at the target's pose read from
# the simulator. Here the arm is aimed at a pose ESTIMATED from what the
# eval camera saw: the detector's box + the camera's depth image.
#
# How it plugs in (the v14 code above is unchanged):
#   1. render_camera() is redefined to also keep the depth buffer and
#      the exact view/projection matrices of the frame it rendered.
#      Same calls in the same order, so scenes and seeds are unchanged.
#   2. attempt_grasp_and_lift() is wrapped. In "estimated" mode it
#      estimates the grasp point and yaw from the detection box + depth,
#      then runs the unchanged v14 grasp aimed at that estimate.
#   3. The estimated-mode trials use the same seeds as the v14 trials,
#      so each trial is a paired comparison (identical scene, identical
#      detection). The v14 results above are reused for the comparison.
#
# What the estimate uses: the detection box, the requested class, the
# eval camera's RGB + depth images and matrices, the table height, and
# fixed fruit-model facts (all fruit rest on the table; the pear's top
# sphere radius and offset; the banana is a chain of equal spheres).
# It does NOT use the segmentation mask or any simulator pose; those are
# read only to LOG the error. Contact/lift checks still read the
# simulator, as in v14 (they stand in for gripper and lift sensors).
#
# One deliberate difference from v14: for a pear that has tipped onto
# its side (about 1 in 5), the fingers close across the pear's axis.
# v14 uses the body's yaw, which has no fixed relation to the axis once
# the pear tips. Each estimated trial logs pose = "upright" / "tipped".
# ================================================================
import pandas as pd

LOC_TRIALS         = 50      # per class, same as v14
NOISE_SEED         = 4004    # own RNG: never touches the scene RNG

CAM_NEAR, CAM_FAR = 0.02, 2.0                 # same as render_camera
TRIAL_OUTCOMES_LOC = "localization_failed"
if TRIAL_OUTCOMES_LOC not in TRIAL_OUTCOMES:
    TRIAL_OUTCOMES.append(TRIAL_OUTCOMES_LOC)  # estimate impossible

# Keep the v14 originals once (safe to re-run this cell).
if "_V14_ORIG" not in globals():
    _V14_ORIG = dict(render_camera=render_camera,
                     attempt=attempt_grasp_and_lift,
                     grasp_point=grasp_point_and_axis,
                     base_pose=p.getBasePositionAndOrientation)

LOCALIZATION_MODE = "ground_truth"   # "ground_truth" (v14) or "estimated"
_LAST_RGBD = {}                      # depth + matrices of the last eval frame
_LOC_LOG   = []                      # one record per estimated grasp attempt
_NOISE_RNG = np.random.default_rng(NOISE_SEED)

# ── 1. Camera: same render as v14, but keep depth + matrices ─────
def render_camera(w=640, h=480, mode="eval"):
    cam_pos, tgt, fov = get_camera_pose(mode)
    view = p.computeViewMatrix(cam_pos.tolist(), tgt.tolist(), [0, 0, 1])
    proj = p.computeProjectionMatrixFOV(fov=fov, aspect=w / h,
                                        nearVal=CAM_NEAR, farVal=CAM_FAR)
    try:
        img = p.getCameraImage(w, h, view, proj,
                               renderer=p.ER_BULLET_HARDWARE_OPENGL)
    except Exception:
        img = p.getCameraImage(w, h, view, proj,
                               renderer=p.ER_TINY_RENDERER)
    rgba  = np.reshape(img[2], (h, w, 4))
    seg   = np.reshape(img[4], (h, w))
    depth = np.reshape(np.asarray(img[3], dtype=np.float64), (h, w))
    rgb   = rgba[:, :, :3].astype(np.uint8)
    _LAST_RGBD.clear()
    _LAST_RGBD.update(depth=depth, view=view, proj=proj, w=w, h=h,
                      rgb=rgb, seg=seg)   # seg kept for logging/plots only
    return rgb, seg

# ── 2. Depth helpers ─────────────────────────────────────────────
def depth_buffer_to_metric(d, near=CAM_NEAR, far=CAM_FAR):
    return far * near / (far - (far - near) * d)

def metric_to_depth_buffer(z, near=CAM_NEAR, far=CAM_FAR):
    return (far - far * near / z) / (far - near)

def add_depth_noise(d):
    """Optional sensor model: Gaussian range noise + random dropouts."""
    if DEPTH_NOISE_STD <= 0 and DEPTH_DROPOUT <= 0:
        return d
    z = depth_buffer_to_metric(d)
    if DEPTH_NOISE_STD > 0:
        z = z + _NOISE_RNG.normal(0.0, DEPTH_NOISE_STD, size=z.shape)
    out = metric_to_depth_buffer(np.clip(z, CAM_NEAR, CAM_FAR * 0.999))
    if DEPTH_DROPOUT > 0:
        out[_NOISE_RNG.random(size=z.shape) < DEPTH_DROPOUT] = 1.0  # no return
    return out

def box_to_points(box, rgbd):
    """Back-project every depth pixel inside a box to world xyz.
    Returns (H_box, W_box, 3) points and a validity mask."""
    H, W = rgbd["h"], rgbd["w"]
    x1, y1, x2, y2 = [int(round(c)) for c in box]
    x1, x2 = max(0, x1), min(W, x2 + 1)
    y1, y2 = max(0, y1), min(H, y2 + 1)
    d = add_depth_noise(rgbd["depth"][y1:y2, x1:x2].copy())
    vs, us = np.mgrid[y1:y2, x1:x2]
    V = np.array(rgbd["view"]).reshape(4, 4, order="F")  # column-major
    P = np.array(rgbd["proj"]).reshape(4, 4, order="F")
    inv = np.linalg.inv(P @ V)
    ndc = np.stack([2 * (us + 0.5) / W - 1,      # pixel -> NDC (row 0 = top)
                    1 - 2 * (vs + 0.5) / H,
                    2 * d - 1,
                    np.ones_like(d)], axis=-1)
    pts = ndc @ inv.T
    pts = pts[..., :3] / pts[..., 3:4]
    valid = d < 0.9999                            # 1.0 = nothing hit
    return pts, valid

def world_to_pixel(xyz, rgbd):
    V = np.array(rgbd["view"]).reshape(4, 4, order="F")
    P = np.array(rgbd["proj"]).reshape(4, 4, order="F")
    c = P @ V @ np.array([*xyz, 1.0])
    c = c[:3] / c[3]
    return ((c[0] + 1) / 2 * rgbd["w"] - 0.5, (1 - c[1]) / 2 * rgbd["h"] - 0.5)

# ── 3. Grasp pose from box + depth ───────────────────────────────
# The estimate mirrors what v14's grasp_point_and_axis() reads from the
# simulator, so the only thing that changes is where the pose comes from:
#   orange -> sphere centre
#   pear   -> centre of the small top sphere
#   banana -> centre of the 3rd sphere from the base end (link 1), and
#             the yaw from the base sphere toward the far end of the chain
#
# Pixels are kept if they are (a) inside the box, (b) above the table
# top and below the arm, and (c) the fruit's colour, measured as colour
# ratios so shading does not matter (a simple stand-in for instance
# segmentation; it drops a neighbouring fruit of another class that
# falls inside a loose box). Shape priors are fixed fruit-model facts:
PEAR_TOP_R_TYPICAL = 0.025      # top sphere radius, spawned 0.022-0.028 m
PEAR_UPRIGHT_MIN_HEIGHT = 0.10  # upright pear >= 0.108 m tall; tipped <= 0.09
BANANA_CAP_BAND      = 0.004    # m below a local height maximum = sphere top
BANANA_SAME_SIZE_TOL = 0.004    # m: caps this close in height = same banana
MIN_OBJECT_POINTS  = 30
CLASS_COLOUR = {                 # (G/R range, max B/R) in the eval frame
    "banana": ((0.80, 1.10), 0.30),
    "orange": ((0.40, 0.70), 0.16),  # B/R cap excludes brown stems (>= 0.12)
    "pear":   ((1.50, 9.99), 9.99),
}
CLASS_MAX_HEIGHT = {             # m above the table: tallest possible instance
    "banana": 0.075,             # sphere radius <= 0.034
    "orange": 0.085,             # radius <= 0.040
    "pear":   0.150,             # upright pear incl. top sphere
}

def _object_points(box, cls, rgbd, table_top_z):
    pts, valid = box_to_points(box, rgbd)
    x1, y1 = max(0, int(round(box[0]))), max(0, int(round(box[1])))
    rgb = rgbd["rgb"][y1:y1 + pts.shape[0], x1:x1 + pts.shape[1]].astype(float) + 1
    (g_lo, g_hi), b_max = CLASS_COLOUR[cls]
    gr, br = rgb[..., 1] / rgb[..., 0], rgb[..., 2] / rgb[..., 0]
    z = pts[..., 2]
    mask = (valid & (z > table_top_z + 0.004)
            & (z < table_top_z + CLASS_MAX_HEIGHT[cls])
            & (gr > g_lo) & (gr < g_hi) & (br < b_max))
    return pts, mask

def _largest_blob(mask):
    n, lab, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    if n < 2:
        return None
    return lab == 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))

def estimate_grasp_pose(box, cls, rgbd, table_top_z):
    """Returns (grasp_point_xyz, axis_yaw, info) or None."""
    if cls not in CLASS_COLOUR:
        return None
    pts, mask = _object_points(box, cls, rgbd, table_top_z)
    if mask.sum() < MIN_OBJECT_POINTS:
        return None
    if cls in ("orange", "pear"):
        blob = _largest_blob(mask)
        P = pts[blob]
        if len(P) < MIN_OBJECT_POINTS:
            return None
        top_z = float(np.percentile(P[:, 2], 99.5))
        yaw, pose = 0.0, "upright"                     # round in x,y: yaw free
        if cls == "orange":
            r   = (top_z - table_top_z) / 2.0         # resting sphere
            cap = P[P[:, 2] > top_z - 0.3 * r]        # cap sits over the centre
            gp  = [*cap[:, :2].mean(axis=0), table_top_z + r]
        elif top_z - table_top_z > PEAR_UPRIGHT_MIN_HEIGHT:
            # upright pear, green pixels only (stem excluded): the highest
            # green point is the top of the small top sphere
            cap = P[P[:, 2] > top_z - 0.3 * PEAR_TOP_R_TYPICAL]
            gp  = [*cap[:, :2].mean(axis=0), top_z - PEAR_TOP_R_TYPICAL]
        else:
            # tipped pear (about 1 in 5 roll onto their side): the highest
            # point is now the top of the BASE sphere. Find the base centre,
            # then the top sphere is the green mass beyond the base radius.
            pose = "tipped"
            rb   = (top_z - table_top_z) / 2.0
            cb   = P[P[:, 2] > top_z - 0.3 * rb][:, :2].mean(axis=0)
            dxy  = P[:, :2] - cb
            far  = dxy[np.linalg.norm(dxy, axis=1) > 1.05 * rb]
            if len(far) < 5:
                return None                           # top sphere hidden
            u    = far.mean(axis=0)
            u    = u / np.linalg.norm(u)
            rt   = PEAR_TOP_R_TYPICAL
            d    = np.sqrt(max((1.32 * rb) ** 2 - (rb - rt) ** 2, 0.0))
            gp   = [*(cb + d * u), table_top_z + rt]  # both spheres on the table
            yaw  = float(np.arctan2(u[1], u[0]))      # fingers close across axis
        return ([float(v) for v in gp], yaw,
                dict(n_points=int(len(P)), n_parts=1, pose=pose))
    # banana: a chain of 5 equal spheres resting on the table, spaced
    # further apart toward the tip. Steps:
    #   a) find each sphere's top cap (a local height maximum in depth)
    #   b) group caps by height: each banana has its own sphere size, so
    #      this separates the target from another banana inside the box
    #   c) keep the group whose spheres best fill the detection box
    #   d) order its caps along the chain, base end = closest-spaced pair,
    #      grasp the 3rd sphere from the base (v14 grasps link 1)
    hgt = np.where(mask, pts[..., 2] - table_top_z, 0.0).astype(np.float32)
    local_max = cv2.dilate(hgt, np.ones((15, 15), np.uint8))
    caps = mask & (hgt >= local_max - BANANA_CAP_BAND)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(caps.astype(np.uint8), 8)
    _, blob_lab = cv2.connectedComponents(mask.astype(np.uint8), connectivity=8)
    found = []
    for k in range(1, n):
        if stats[k, cv2.CC_STAT_AREA] < 3:
            continue
        sel = lab == k
        blob_ids = np.unique(blob_lab[sel])
        found.append(dict(xy=pts[sel][:, :2].mean(axis=0),
                          h=float(hgt[sel].max()),
                          blob=np.isin(blob_lab, blob_ids[blob_ids > 0])))
    if len(found) < 3:
        return None
    found.sort(key=lambda c: c["h"])
    groups, cur = [], [found[0]]
    for c in found[1:]:
        if c["h"] - cur[-1]["h"] > BANANA_SAME_SIZE_TOL:
            groups.append(cur); cur = [c]
        else:
            cur.append(c)
    groups.append(cur)
    bh, bw = mask.shape
    def fill_iou(g):                    # how well the group fills the box
        ys, xs = np.where(np.logical_or.reduce([c["blob"] for c in g]))
        gx1, gy1, gx2, gy2 = xs.min(), ys.min(), xs.max() + 1, ys.max() + 1
        return (gx2 - gx1) * (gy2 - gy1) / float(bw * bh)
    groups = [g for g in groups if len(g) >= 3]
    if not groups:
        return None
    g = max(groups, key=lambda g: (round(fill_iou(g), 2), len(g)))
    r = float(np.median([c["h"] for c in g])) / 2.0
    C = np.array([c["xy"] for c in g])
    mu = C.mean(axis=0)
    _, _, vt = np.linalg.svd(C - mu, full_matrices=False)
    C = C[np.argsort((C - mu) @ vt[0])]                    # order along chain
    if np.linalg.norm(C[1] - C[0]) > np.linalg.norm(C[-1] - C[-2]):
        C = C[::-1]                                        # base end first
    gxy = C[2]                                             # 3rd sphere = link 1
    yaw = float(np.arctan2(C[-1][1] - C[0][1], C[-1][0] - C[0][0]))
    return ([float(gxy[0]), float(gxy[1]), table_top_z + r], yaw,
            dict(n_points=int(mask.sum()), n_parts=int(len(C)),
                 n_banana_groups=len(groups)))

def yaw_error_deg(a, b):
    """Gripper is symmetric under 180 deg."""
    d = (a - b + np.pi / 2) % np.pi - np.pi / 2
    return float(abs(np.degrees(d)))

def log_localization(obj_id, cls, est, extra=None):
    true_gp, true_yaw = _V14_ORIG["grasp_point"](obj_id)     # logging only
    rec = dict(cls=cls, true_gp=[round(v, 4) for v in true_gp])
    if est is None:
        rec.update(est_gp=None, err_xy_cm=None, err_z_cm=None, yaw_err_deg=None)
    else:
        gp, yaw, info = est
        rec.update(est_gp=[round(v, 4) for v in gp],
                   err_xy_cm=round(100 * float(np.hypot(gp[0] - true_gp[0],
                                                        gp[1] - true_gp[1])), 2),
                   err_z_cm=round(100 * (gp[2] - true_gp[2]), 2),
                   yaw_err_deg=(round(yaw_error_deg(yaw, true_yaw), 1)
                                if cls == "banana" else None),
                   **info)
    if extra:
        rec.update(extra)
    return rec

# ── 4. Grasp wrapper: aim the unchanged v14 grasp at the estimate ─
def attempt_grasp_and_lift(robot_id, arm_joints, finger_joints, ee_link,
                           obj_id, table_top_z, lift_threshold,
                           writer=None, vid_w=640, vid_h=480,
                           detections=None, grasp_mode=None):
    args = (robot_id, arm_joints, finger_joints, ee_link,
            obj_id, table_top_z, lift_threshold)
    kw = dict(writer=writer, vid_w=vid_w, vid_h=vid_h,
              detections=detections, grasp_mode=grasp_mode)
    if LOCALIZATION_MODE != "estimated" or not detections:
        return _V14_ORIG["attempt"](*args, **kw)
    conf, cls, box = detections[0]
    est = estimate_grasp_pose(box, cls, _LAST_RGBD, table_top_z)
    _LOC_LOG.append(log_localization(obj_id, cls, est))
    if est is None:
        pos, _ = _V14_ORIG["base_pose"](obj_id)
        return False, pos[2] - table_top_z, "localization_failed"
    gp, yaw, _ = est
    # v14's grasp reads the target pose from the simulator twice: the
    # object's base pose (first line: used for the lift x,y) and
    # grasp_point_and_axis(). Both are replaced by the estimate for this
    # call only. Later pose reads (contact check, constraint, final lift
    # height) stay real: they are physics, not perception.
    first = {"done": False}
    est_pose = (tuple(gp), p.getQuaternionFromEuler([0, 0, yaw]))
    def _pose(body, *a, **k):
        if not first["done"] and body == obj_id:
            first["done"] = True
            return est_pose
        return _V14_ORIG["base_pose"](body, *a, **k)
    g = _V14_ORIG["attempt"].__globals__
    g["grasp_point_and_axis"] = lambda _oid: (list(gp), yaw)
    p.getBasePositionAndOrientation = _pose
    try:
        return _V14_ORIG["attempt"](*args, **kw)
    finally:
        p.getBasePositionAndOrientation = _V14_ORIG["base_pose"]
        g["grasp_point_and_axis"] = _V14_ORIG["grasp_point"]

# ── 5. Validation: estimate vs. truth on ground-truth boxes ──────
def validate_localization(n_scenes=10, seed=5005):
    """Uses the GT box (padded like the training labels) in place of a
    detection, so this checks the depth math alone. Expect errors of
    about 1 cm; much more usually means a coordinate bug."""
    seed_everything(seed)
    connect_direct()
    recs = []
    for i, cls in enumerate(CFG.fruit_classes):
        got = 0
        while got < n_scenes:
            scene = build_trial_scene(cls, CFG.fruits_in_scene)
            outcome, rgb, seg, gtb, _ = pre_detection_gate(scene)
            tfid = scene["target_fid"]
            if outcome is not None or gtb.get(tfid) is None:
                continue
            box = pad_bbox(gtb[tfid], CFG.bbox_pad, CFG.cam_w, CFG.cam_h)
            est = estimate_grasp_pose(box, cls, _LAST_RGBD, scene["table_top_z"])
            recs.append(log_localization(tfid, cls, est))
            if got == 0 and est is not None:
                _plot_example(cls, box, est, tfid)
            got += 1
    p.disconnect()
    df = pd.DataFrame(recs)
    print("\n=== Localization check on GT boxes (no detector) ===")
    a = df[["cls", "err_xy_cm", "err_z_cm", "yaw_err_deg"]].copy()
    a[["err_xy_cm", "err_z_cm", "yaw_err_deg"]] = a[["err_xy_cm", "err_z_cm",
                                                     "yaw_err_deg"]].abs()
    print(a.groupby("cls").agg(["median", "max"]).round(2))
    print("failed estimates:", int(df["est_gp"].isna().sum()))
    if a["err_xy_cm"].median() > 2.0:
        print("WARNING: errors well above ~1 cm usually mean a coordinate "
              "bug (matrix order, flipped rows, wrong resolution).")
    return df

_EXAMPLE_FIGS = []
def _plot_example(cls, box, est, tfid):
    """Saves a figure for the report: RGB + box, depth, est vs. true point."""
    import matplotlib.pyplot as plt
    rgbd = _LAST_RGBD
    gp, yaw, _ = est
    true_gp, _ = _V14_ORIG["grasp_point"](tfid)
    ue, ve = world_to_pixel(gp, rgbd)
    ut, vt = world_to_pixel(true_gp, rgbd)
    zm = depth_buffer_to_metric(rgbd["depth"])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    ax[0].imshow(rgbd["rgb"]); ax[1].imshow(zm, cmap="viridis_r", vmin=0.4, vmax=1.2)
    for a in ax:
        x1, y1, x2, y2 = box
        a.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                  ec="red", lw=1.5))
        a.plot(ut, vt, "wo", ms=7, mec="k", label="true grasp point")
        a.plot(ue, ve, "r+", ms=12, mew=2, label="estimated from depth")
        a.axis("off")
    ax[0].set_title(f"{cls}: RGB + box"); ax[1].set_title("depth (m)")
    ax[0].legend(loc="lower left", fontsize=8)
    os.makedirs(CFG.results_dir, exist_ok=True)
    path = os.path.join(CFG.results_dir, f"localization_example_{cls}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    _EXAMPLE_FIGS.append(path)
    plt.show()

# ── 6. Paired end-to-end trials ──────────────────────────────────
def run_localization_trials(trials=LOC_TRIALS, model=None, class_names=None):
    global LOCALIZATION_MODE
    if model is None:
        model, class_names = load_detector(CFG.model_path, CFG.classes_path)
    noise_tag = ("ideal" if DEPTH_NOISE_STD <= 0 and DEPTH_DROPOUT <= 0
                 else f"noise{DEPTH_NOISE_STD*1000:g}mm_drop{DEPTH_DROPOUT:g}")
    table, all_rows = [], {}
    for i, cls in enumerate(CFG.fruit_classes):
        seed = CFG.trial_seed + i                 # same seeds as v14
        res = {}
        for mode in ["ground_truth", "estimated"]:
            LOCALIZATION_MODE = mode
            _LOC_LOG.clear()
            if mode == "ground_truth":
                # same tag as the v14 trials above: reuse them if saved
                tag = f"trials_{cls}_{CFG.grasp_mode}_seed{seed}"
                saved = os.path.join(CFG.results_dir, tag + ".json")
                if os.path.exists(saved):
                    d = json.load(open(saved))
                    if len(d["rows"]) == trials:
                        print(f"[REUSE] {saved}")
                        res[mode] = (d["summary"], d["rows"])
                        continue
            else:
                tag = f"trials_{cls}_{CFG.grasp_mode}_seed{seed}_depth_{noise_tag}"
                saved = os.path.join(CFG.results_dir, tag + ".json")
                if os.path.exists(saved):
                    d = json.load(open(saved))
                    if len(d["rows"]) == trials and "localization" in d["summary"]:
                        print(f"[REUSE] {saved}")
                        res[mode] = (d["summary"], d["rows"])
                        continue
            summary, rows = run_trials(
                requested=cls, trials=trials, seed=seed, tag=tag,
                video_path=(f"successful_grasp_{cls}.mp4" if mode == "ground_truth"
                            else f"successful_grasp_{cls}_depth.mp4"),
                model=model, class_names=class_names)
            if mode == "estimated":
                # attach the localization record to each grasp-stage trial
                stage = [r for r in rows if r["outcome"] in
                         ("grasp_missed", "lift_insufficient", "success",
                          TRIAL_OUTCOMES_LOC)]
                assert len(stage) == len(_LOC_LOG), "log/trial mismatch"
                for r, rec in zip(stage, _LOC_LOG):
                    r.update({k: rec.get(k) for k in
                              ("est_gp", "true_gp", "err_xy_cm",
                               "err_z_cm", "yaw_err_deg", "pose")})
                errs = [r["err_xy_cm"] for r in stage if r.get("err_xy_cm") is not None]
                summary["localization"] = dict(
                    depth=noise_tag,
                    median_err_xy_cm=float(np.median(errs)) if errs else None,
                    p90_err_xy_cm=float(np.percentile(errs, 90)) if errs else None)
                save_results(tag, summary, rows)
            res[mode] = (summary, rows)
        LOCALIZATION_MODE = "ground_truth"
        (s_gt, r_gt), (s_es, r_es) = res["ground_truth"], res["estimated"]
        ok_gt = [r["outcome"] == "success" for r in r_gt]
        ok_es = [r["outcome"] == "success" for r in r_es]
        same_scene = all(a["target_pixels"] == b["target_pixels"]
                         for a, b in zip(r_gt, r_es))
        loc = s_es["localization"]
        table.append(dict(
            cls=cls, trials=trials, depth=noise_tag,
            success_sim_pose=sum(ok_gt), success_depth_est=sum(ok_es),
            both=sum(a and b for a, b in zip(ok_gt, ok_es)),
            sim_pose_only=sum(a and not b for a, b in zip(ok_gt, ok_es)),
            depth_est_only=sum(b and not a for a, b in zip(ok_gt, ok_es)),
            localization_failed=s_es["counts"].get(TRIAL_OUTCOMES_LOC, 0),
            median_err_xy_cm=loc["median_err_xy_cm"],
            p90_err_xy_cm=loc["p90_err_xy_cm"],
            paired_scenes_identical=same_scene))
        all_rows[cls] = res
    df = pd.DataFrame(table)
    df.to_csv(os.path.join(CFG.results_dir,
                           f"summary_localization_{noise_tag}.csv"), index=False)
    print("\n=== End-to-end success: simulator pose vs. depth estimate ===")
    display(df)
    return df, all_rows

# ── RUN ──────────────────────────────────────────────────────────
if RUN_LOC_VALIDATION:
    loc_check_df = validate_localization()
if RUN_LOC_TRIALS:
    loc_df, loc_rows = run_localization_trials()